In [2]:
import os
import re
import time
import json
import torch
import commons
import utils
from models import SynthesizerTrn
from text_JP import cleaned_text_to_sequence, symbols
import pyopenjtalk
from text_JP.phonemize import Phonemizer

# This script is designed to be run in a Jupyter Notebook or an environment
# with IPython display capabilities.
from IPython.display import Audio, display

# 1 全体デコード 

In [10]:
# --- 1. Configuration (EDIT THESE PATHS) ---
# ===========================================
# Path to the config file of the trained multi-speaker model
config_path = "./logs/uudb_csj21/config.json" 

# Path to the generator checkpoint of the trained model
# Find the latest G_****.pth file in your model directory
checkpoint_path = "./logs/uudb_csj21/G_3020000.pth" # <-- IMPORTANT: UPDATE THIS PATH

# Text to be synthesized
text_to_synthesize = "最近、インターステラーを見たのですけど、すごく面白かったです。"
# ===========================================


# --- 2. Text Pre-processing Functions ---
def japanese_cleaner_revised(text):
    parts = re.split(r'({cough}|<cough>|\[.*?\]|[、。])', text)
    phoneme_parts = []
    phonemizer = Phonemizer()
    for part in parts:
        if not part or part.isspace():
            continue
        if part.startswith('[') and part.endswith(']') and len(part) > 2:
            content = part[1:-1]
            if not content:
                phoneme_parts.append('[ ]')
            else:
                kana_content = pyopenjtalk.g2p(content, kana=True).replace('ヲ', 'オ')
                phoneme_content = phonemizer(kana_content)
                phoneme_parts.append(f'[ {phoneme_content} ]')
            continue
        if part == '{cough}' or part == '<cough>':
            phoneme_parts.append('<cough>')
            continue
        if part in '、。':
            phoneme_parts.append('sp')
            continue
        kana = pyopenjtalk.g2p(part, kana=True).replace('ヲ', 'オ')
        phonemes = phonemizer(kana)
        phoneme_parts.append(phonemes)
    final_text = ' '.join(phoneme_parts)
    return re.sub(r'\s+', ' ', final_text).strip()

def text_to_sequence_custom(text, hps):
    phonemized_text = japanese_cleaner_revised(text)
    stn_tst = cleaned_text_to_sequence(phonemized_text)
    if hps.data.add_blank:
        stn_tst = commons.intersperse(stn_tst, 0)
    return torch.LongTensor(stn_tst)


# --- 3. Main Synthesis Process ---
if not os.path.exists(config_path):
    print(f"ERROR: Config file not found at {config_path}")
elif not os.path.exists(checkpoint_path):
    print(f"ERROR: Checkpoint file not found at {checkpoint_path}")
    print("Please update the 'checkpoint_path' variable in this script to point to your trained model.")
else:
    # Load configuration
    hps = utils.get_hparams_from_file(config_path)

    # Determine device
    #device = "cuda" if torch.cuda.is_available() else "cpu"
    device = "cpu"

    # Load model
    print("Loading model...")
    net_g = SynthesizerTrn(
        len(symbols),
        hps.data.filter_length // 2 + 1,
        hps.train.segment_size // hps.data.hop_length,
        n_speakers=hps.data.n_speakers,
        **hps.model).to(device)
    
    # Set model to evaluation mode
    _ = net_g.eval()
    
    # Load checkpoint
    print(f"Loading checkpoint from {checkpoint_path}...")
    _ = utils.load_checkpoint(checkpoint_path, net_g, None)

    # Process text
    print(f"Original text: {text_to_synthesize}")

    stn_tst = text_to_sequence_custom(text_to_synthesize, hps)

    i=375
    # Synthesize for each speaker
    print(f"\n--- Synthesizing for Speaker {i} ---")
    start_time = time.time()
        
    with torch.no_grad():
        x_tst = stn_tst.unsqueeze(0).to(device)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        sid = torch.LongTensor([i]).to(device)
            
        # Inference
        audio = net_g.infer(x_tst, x_tst_lengths, sid=sid, 
                            noise_scale=0.1, 
                            noise_scale_w=1.0, 
                            length_scale=1.0
                            )[0][0,0].data.cpu().float().numpy()

    end_time = time.time()
    elapsed_time = end_time - start_time
    audio_duration = len(audio) / hps.data.sampling_rate
    rtf = elapsed_time / audio_duration

    print(f"Audio duration: {audio_duration:.2f} seconds")
    print(f"Elapsed time: {elapsed_time:.2f} seconds")
    print(f"Real Time Factor (RTF): {rtf:.4f}")
    display(Audio(audio, rate=hps.data.sampling_rate, normalize=False))

    print("\nSynthesis complete.")



Loading model...
Mutli-stream iSTFT VITS
Loading checkpoint from ./logs/uudb_csj21/G_3020000.pth...
Original text: 最近、インターステラーを見たのですけど、すごく面白かったです。

--- Synthesizing for Speaker 375 ---
Audio duration: 4.18 seconds
Elapsed time: 0.21 seconds
Real Time Factor (RTF): 0.0504



Synthesis complete.


# 2 細切れデコード　overlapなし 

In [24]:
if not os.path.exists(config_path):
    print(f"ERROR: Config file not found at {config_path}")
elif not os.path.exists(checkpoint_path):
    print(f"ERROR: Checkpoint file not found at {checkpoint_path}")
    print("Please update the 'checkpoint_path' variable in this script to point to your trained model.")
else:
    # Load configuration
    hps = utils.get_hparams_from_file(config_path)

    # Determine device
    # device = "cuda" if torch.cuda.is_available() else "cpu"
    device = "cpu" # 必要に応じてcudaに変更してください

    # Load model
    print("Loading model...")
    net_g = SynthesizerTrn(
        len(symbols),
        hps.data.filter_length // 2 + 1,
        hps.train.segment_size // hps.data.hop_length,
        n_speakers=hps.data.n_speakers,
        **hps.model).to(device)
    
    _ = net_g.eval()
    
    print(f"Loading checkpoint from {checkpoint_path}...")
    _ = utils.load_checkpoint(checkpoint_path, net_g, None)

    print(f"Original text: {text_to_synthesize}")
    stn_tst = text_to_sequence_custom(text_to_synthesize, hps)

    # Speaker ID
    i = 375 # 任意のスピーカーID
    print(f"\n--- Synthesizing for Speaker {i} (Chunk Decoding) ---")
    
    with torch.no_grad():
        x_tst = stn_tst.unsqueeze(0).to(device)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        sid = torch.LongTensor([i]).to(device)
            
        # 1. まず潜在表現 z を全体生成 (infer_z_only を使用)
        # ---------------------------------------------------------
        start_time_z = time.time()
        
        # infer_z_only は (attn, y_mask, (z, z_p, m_p, logs_p), timings) を返します
        # 必要なのは z (タプルの0番目) と y_mask です
        attn, y_mask, (z, z_p, m_p, logs_p), timings = net_g.infer_z_only(
            x_tst, 
            x_tst_lengths, 
            sid=sid, 
            noise_scale=0.1, 
            noise_scale_w=1.0, 
            length_scale=1.0
        )
        
        # z は padding されている可能性があるため、y_mask でマスク処理をしておきます
        z = z * y_mask
        
        time_z = time.time() - start_time_z
        print(f"Latent Z generation time: {time_z:.4f} sec")
        print(f"Latent Z shape: {z.shape}") # [Batch, Channels, Time]

        # 2. Decoder 用の Speaker Embedding (g) を準備
        # ---------------------------------------------------------
        # infer_z_only は g を返さないため、ここで手動生成します
        if net_g.n_speakers > 0:
            g = net_g.emb_g(sid).unsqueeze(-1)
        else:
            g = None

        # 3. 10フレームごとに分割してデコード (Chunk Decoding)
        # ---------------------------------------------------------
        chunk_size = 10  # 1セグメントあたりのフレーム数
        full_audio_chunks = []
        
        z_channels, z_time_length = z.shape[1], z.shape[2]
        
        print(f"Start decoding in chunks of size {chunk_size}...")
        start_time_dec = time.time()

        for step in range(0, z_time_length, chunk_size):
            # z をスライス (最後の端数も自動的に処理されます)
            end_step = min(step + chunk_size, z_time_length)
            z_chunk = z[:, :, step:end_step]
            
            # 部分デコード
            # models.py の dec は (o, o_mb, spec, phase) を返します
            o_chunk, _, _, _ = net_g.dec(z_chunk, g=g)
            
            full_audio_chunks.append(o_chunk)

        # 4. 結合して最終的な音声にする
        # ---------------------------------------------------------
        full_audio = torch.cat(full_audio_chunks, dim=2)
        
        end_time = time.time()
        
        # 結果の整形
        audio = full_audio[0, 0].data.cpu().float().numpy()

    # パフォーマンス計測
    elapsed_time = end_time - start_time_z
    audio_duration = len(audio) / hps.data.sampling_rate
    rtf = elapsed_time / audio_duration

    print(f"Total Audio duration: {audio_duration:.2f} seconds")
    print(f"Total Elapsed time: {elapsed_time:.2f} seconds")
    print(f"Real Time Factor (RTF): {rtf:.4f}")
    
    # 音声再生
    display(Audio(audio, rate=hps.data.sampling_rate, normalize=False))

    print("\nSynthesis complete.")

Loading model...
Mutli-stream iSTFT VITS
Loading checkpoint from ./logs/uudb_csj21/G_3020000.pth...
Original text: 最近、インターステラーを見たのですけど、すごく面白かったです

--- Synthesizing for Speaker 375 (Chunk Decoding) ---
Latent Z generation time: 0.0245 sec
Latent Z shape: torch.Size([1, 192, 252])
Start decoding in chunks of size 10...
Total Audio duration: 4.03 seconds
Total Elapsed time: 0.31 seconds
Real Time Factor (RTF): 0.0762



Synthesis complete.


# 3 細切れデコード　overlapあり

In [25]:
# ... (前略: imports, functions, config設定などはそのまま) ...

# --- 3. Main Synthesis Process ---
if not os.path.exists(config_path):
    print(f"ERROR: Config file not found at {config_path}")
elif not os.path.exists(checkpoint_path):
    print(f"ERROR: Checkpoint file not found at {checkpoint_path}")
else:
    # Load configuration
    hps = utils.get_hparams_from_file(config_path)

    # Determine device
    device = "cpu" # 必要に応じて "cuda"

    # Load model
    print("Loading model...")
    net_g = SynthesizerTrn(
        len(symbols),
        hps.data.filter_length // 2 + 1,
        hps.train.segment_size // hps.data.hop_length,
        n_speakers=hps.data.n_speakers,
        **hps.model).to(device)
    
    _ = net_g.eval()
    
    print(f"Loading checkpoint from {checkpoint_path}...")
    _ = utils.load_checkpoint(checkpoint_path, net_g, None)

    print(f"Original text: {text_to_synthesize}")
    stn_tst = text_to_sequence_custom(text_to_synthesize, hps)

    # Speaker ID
    i = 375
    print(f"\n--- Synthesizing for Speaker {i} (Overlap-Add Decoding) ---")
    
    with torch.no_grad():
        x_tst = stn_tst.unsqueeze(0).to(device)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        sid = torch.LongTensor([i]).to(device)
            
        # 1. 潜在表現 z の全体生成
        # ---------------------------------------------------------
        attn, y_mask, (z, z_p, m_p, logs_p), timings = net_g.infer_z_only(
            x_tst, 
            x_tst_lengths, 
            sid=sid, 
            noise_scale=0.1, 
            noise_scale_w=1.0, 
            length_scale=1.0
        )
        z = z * y_mask

        # Decoder用 Speaker Embedding
        if net_g.n_speakers > 0:
            g = net_g.emb_g(sid).unsqueeze(-1)
        else:
            g = None

        # 2. Overlap-Add の設定
        # ---------------------------------------------------------
        z_frame_window = 10     # z上のウィンドウサイズ (フレーム数)
        z_frame_hop = 5         # z上のホップサイズ (フレーム数)
        
        # 1フレームあたりの音声サンプル数 (アップサンプリング率)
        upsample_factor = hps.data.hop_length 
        
        # 音声波形上でのウィンドウサイズとホップサイズ
        audio_window_size = z_frame_window * upsample_factor
        
        # Hann窓の作成
        # shape: [1, 1, audio_window_size]
        window = torch.hann_window(audio_window_size).to(device).view(1, 1, -1)
        
        # 出力バッファの準備
        z_total_frames = z.shape[2]
        total_audio_len = z_total_frames * upsample_factor
        
        # 加算用バッファ (音声波形)
        y_buffer = torch.zeros(1, 1, total_audio_len).to(device)
        # 正規化用バッファ (窓の重みの合計)
        w_buffer = torch.zeros(1, 1, total_audio_len).to(device)

        print(f"Total Z frames: {z_total_frames}")
        print(f"Audio Window Size: {audio_window_size} samples")
        
        # 3. ループ処理 (Overlap-Add)
        # ---------------------------------------------------------
        start_time_dec = time.time()

        for idx in range(0, z_total_frames, z_frame_hop):
            # z の切り出し範囲
            z_start = idx
            z_end = min(idx + z_frame_window, z_total_frames)
            
            # z をスライス
            z_chunk = z[:, :, z_start:z_end]
            
            # 端数処理: 最後のチャンクがウィンドウサイズより小さい場合
            current_z_len = z_chunk.shape[2]
            if current_z_len == 0:
                break
                
            # 部分デコード
            # output shape: [1, 1, current_z_len * upsample_factor]
            o_chunk, _, _, _ = net_g.dec(z_chunk, g=g)
            
            # 音声バッファ上の配置位置
            audio_start = z_start * upsample_factor
            audio_end = audio_start + o_chunk.shape[2]
            
            # 現在のチャンク長に対応する窓を取得
            # (最後尾など、フルサイズでない場合に対応)
            current_audio_len = o_chunk.shape[2]
            current_window = window[:, :, :current_audio_len]
            
            # バッファに加算 (音声 * 窓)
            # ※ バッファからはみ出さないようにサイズチェック
            valid_len = min(current_audio_len, y_buffer.shape[2] - audio_start)
            
            y_buffer[:, :, audio_start : audio_start + valid_len] += (o_chunk[:, :, :valid_len] * current_window[:, :, :valid_len])
            w_buffer[:, :, audio_start : audio_start + valid_len] += current_window[:, :, :valid_len]

        # 4. 正規化 (重み付け平均をとる)
        # ---------------------------------------------------------
        # 重みが0の部分(無音部)でのゼロ除算を防ぐため小さい値を足す
        final_audio_tensor = y_buffer / (w_buffer + 1e-8)
        
        end_time = time.time()
        
        # Numpy変換
        audio = final_audio_tensor[0, 0].data.cpu().float().numpy()

    # 結果表示
    elapsed_time = end_time - start_time_dec # デコード部分のみの時間
    audio_duration = len(audio) / hps.data.sampling_rate
    
    print(f"Total Audio duration: {audio_duration:.2f} seconds")
    print(f"Decoding Elapsed time: {elapsed_time:.2f} seconds")
    
    display(Audio(audio, rate=hps.data.sampling_rate, normalize=False))
    print("\nSynthesis complete.")

Loading model...
Mutli-stream iSTFT VITS
Loading checkpoint from ./logs/uudb_csj21/G_3020000.pth...
Original text: 最近、インターステラーを見たのですけど、すごく面白かったです

--- Synthesizing for Speaker 375 (Overlap-Add Decoding) ---
Total Z frames: 252
Audio Window Size: 2560 samples
Total Audio duration: 4.03 seconds
Decoding Elapsed time: 0.56 seconds



Synthesis complete.


# 4 細切れデコード　overlapあり　相互相関で調整

In [28]:
# ... (前略: imports, functions, config設定などはそのまま) ...

import torch.nn.functional as F

# --- 相互相関によるオフセット計算関数 ---
def find_best_shift(ref_segment, target_segment, search_range=100):
    """
    ref_segment: 前の波形の末尾 (1, 1, T)
    target_segment: 次の波形の先頭 (1, 1, T)
    search_range: 探索するラグの範囲 (+/- samples)
    
    戻り値: 最適なシフト量 (正の値ならtargetを後ろにずらす、負なら前にずらす)
    """
    # conv1d で相互相関を計算するために次元を調整
    # ref (filter) shape: (Out_ch, In_ch/Groups, Kernel) -> (1, 1, T)
    # target (input) shape: (1, 1, T + padding)
    
    # ターゲット側にパディングを入れて探索範囲を作る
    # ターゲットを少し広く取って、リファレンスがどこにマッチするか探すイメージ
    pad_target = F.pad(target_segment, (search_range, search_range))
    
    # Conv1dで相互相関を計算 (refをカーネルとする)
    # refは反転させる必要はない（相関を求めたいのでそのまま畳み込む）
    cross_corr = F.conv1d(pad_target, ref_segment)
    
    # 相関が最大になるインデックスを取得
    max_idx = torch.argmax(cross_corr)
    
    # インデックスをシフト量に変換
    # max_idx が search_range のときシフト0
    shift = max_idx.item() - search_range
    
    return shift

# --- 3. Main Synthesis Process ---
if not os.path.exists(config_path):
    print(f"ERROR: Config file not found at {config_path}")
elif not os.path.exists(checkpoint_path):
    print(f"ERROR: Checkpoint file not found at {checkpoint_path}")
else:
    # Load configuration
    hps = utils.get_hparams_from_file(config_path)
    device = "cpu" # 必要に応じて "cuda"

    # Load model
    print("Loading model...")
    net_g = SynthesizerTrn(
        len(symbols),
        hps.data.filter_length // 2 + 1,
        hps.train.segment_size // hps.data.hop_length,
        n_speakers=hps.data.n_speakers,
        **hps.model).to(device)
    
    _ = net_g.eval()
    _ = utils.load_checkpoint(checkpoint_path, net_g, None)

    print(f"Original text: {text_to_synthesize}")
    stn_tst = text_to_sequence_custom(text_to_synthesize, hps)

    # Speaker ID
    i = 375
    print(f"\n--- Synthesizing for Speaker {i} (Correlation-based OLA) ---")
    
    with torch.no_grad():
        x_tst = stn_tst.unsqueeze(0).to(device)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        sid = torch.LongTensor([i]).to(device)
            
        # 1. 潜在表現 z の生成
        attn, y_mask, (z, z_p, m_p, logs_p), timings = net_g.infer_z_only(
            x_tst, x_tst_lengths, sid=sid, noise_scale=0.1, noise_scale_w=1.0, length_scale=1.0
        )
        z = z * y_mask

        if net_g.n_speakers > 0:
            g = net_g.emb_g(sid).unsqueeze(-1)
        else:
            g = None

        # 2. パラメータ設定
        # ---------------------------------------------------------
        z_chunk_size = 10     # デコードするzの長さ
        z_hop_size = 5        # 次のチャンクへ進むzの長さ
        
        upsample_factor = hps.data.hop_length 
        
        # 音声波形上での理論的なオーバーラップ長
        overlap_len = (z_chunk_size - z_hop_size) * upsample_factor
        
        # 相互相関の探索範囲 (サンプル数)
        # 大きすぎると誤検知する可能性があるため、ピッチ周期の半分程度(～100程度)が適当
        search_range = 100 
        
        # 全フレーム数
        z_total_frames = z.shape[2]
        
        # 結果格納用バッファ（逐次結合していく）
        full_audio_tensor = torch.zeros(1, 1, 0).to(device)

        print(f"Overlap Length: {overlap_len} samples")
        print(f"Search Range: +/- {search_range} samples")

        # 3. ループ処理
        # ---------------------------------------------------------
        start_time_dec = time.time()
        
        # 最初のチャンク処理用フラグ
        is_first_chunk = True

        for idx in range(0, z_total_frames, z_hop_size):
            # z の切り出し
            z_end = min(idx + z_chunk_size, z_total_frames)
            z_chunk = z[:, :, idx:z_end]
            
            # デコード
            o_chunk, _, _, _ = net_g.dec(z_chunk, g=g)
            
            # --- 結合処理 ---
            if is_first_chunk:
                # 最初はそのまま採用
                full_audio_tensor = o_chunk
                is_first_chunk = False
            else:
                # 前回の音声の末尾（オーバーラップ対象部分）
                # 探索範囲のために少し余分に取っておくことはせず、純粋なオーバーラップ領域比較とする
                prev_tail = full_audio_tensor[:, :, -overlap_len:]
                
                # 今回の音声の先頭（オーバーラップ対象部分）
                curr_head = o_chunk[:, :, :overlap_len]
                
                # 相互相関で最適なシフト量を計算
                # サイズが足りない場合（最後尾など）はシフト計算をスキップ
                if prev_tail.shape[2] == overlap_len and curr_head.shape[2] == overlap_len:
                    shift = find_best_shift(prev_tail, curr_head, search_range)
                else:
                    shift = 0
                
                # シフト適用後の今回のチャンク
                # shift > 0: 今回の波形を右（未来）にずらす -> 先頭を削る
                # shift < 0: 今回の波形を左（過去）にずらす -> 先頭にパディング or 前回の波形を削る
                # 簡易化のため、今回の波形の読み出し位置（開始点）を調整します
                
                # ベースの開始位置
                start_idx = 0
                
                # 最適位置への補正
                # ここでは「今回の波形をどこから使い始めるか」で調整
                adjusted_start = start_idx - shift
                
                # インデックスが範囲外にならないようクリップ
                adjusted_start = max(0, min(adjusted_start, search_range * 2))
                
                # 位置合わせ済みの新しいチャンク
                aligned_chunk = o_chunk[:, :, adjusted_start:]
                
                # クロスフェード用の長さ
                cross_fade_len = min(overlap_len, aligned_chunk.shape[2])
                
                # 窓関数作成 (Hann窓)
                fade_out = torch.linspace(1.0, 0.0, cross_fade_len).to(device).view(1, 1, -1)
                fade_in = torch.linspace(0.0, 1.0, cross_fade_len).to(device).view(1, 1, -1)
                
                # 前回の波形の末尾と、今回の波形の先頭をクロスフェード
                # 前回の波形を上書きする形で合成
                
                # 1. 前回の波形のフェードアウト部分
                prev_overlap_part = full_audio_tensor[:, :, -cross_fade_len:] * fade_out
                
                # 2. 今回の波形のフェードイン部分
                curr_overlap_part = aligned_chunk[:, :, :cross_fade_len] * fade_in
                
                # 3. 合成部分
                merged_part = prev_overlap_part + curr_overlap_part
                
                # 4. バッファ更新
                # (前回の末尾を削って合成部分に置き換え + 今回の残り部分を追加)
                full_audio_tensor = torch.cat([
                    full_audio_tensor[:, :, :-cross_fade_len], # 重ならない既存部分
                    merged_part,                               # クロスフェード部分
                    aligned_chunk[:, :, cross_fade_len:]       # 新規部分
                ], dim=2)

            # 進捗チェック（最後尾で終了）
            if z_end == z_total_frames:
                break

        end_time = time.time()
        
        audio = full_audio_tensor[0, 0].data.cpu().float().numpy()

    # 結果表示
    elapsed_time = end_time - start_time_dec
    audio_duration = len(audio) / hps.data.sampling_rate
    
    print(f"Total Audio duration: {audio_duration:.2f} seconds")
    print(f"Decoding Elapsed time: {elapsed_time:.2f} seconds")
    
    display(Audio(audio, rate=hps.data.sampling_rate, normalize=False))
    print("\nSynthesis complete.")

Loading model...
Mutli-stream iSTFT VITS
Original text: 最近、インターステラーを見たのですけど、すごく面白かったです

--- Synthesizing for Speaker 375 (Correlation-based OLA) ---
Overlap Length: 1280 samples
Search Range: +/- 100 samples
Total Audio duration: 4.01 seconds
Decoding Elapsed time: 0.79 seconds



Synthesis complete.


# 5 細切れデコード　スペクトログラムをoverlapで接合　相互相関で調整はしない

In [38]:
import os
import time
import numpy as np
import torch
import torch.nn.functional as F
import utils
from models import SynthesizerTrn
from text_JP import cleaned_text_to_sequence, symbols
from IPython.display import Audio, display

# ==========================================
# 1. 必要なクラス定義 (TorchSTFT)
# ==========================================
class TorchSTFT(torch.nn.Module):
    def __init__(self, filter_length=800, hop_length=200, win_length=800, window='hann'):
        super().__init__()
        self.filter_length = filter_length
        self.hop_length = hop_length
        self.win_length = win_length
        if window == 'hann':
            self.window = torch.hann_window(win_length, periodic=True)
        else:
            self.window = torch.hann_window(win_length, periodic=True)

    def inverse(self, magnitude, phase):
        complex_spec = magnitude * torch.exp(phase * 1j)
        inverse_transform = torch.istft(
            complex_spec,
            self.filter_length, self.hop_length, self.win_length, 
            window=self.window.to(complex_spec.device)
        )
        return inverse_transform.unsqueeze(1)

# ==========================================
# 2. メイン合成プロセス (Fixed Ratio OLA)
# ==========================================

# 設定（パスは適宜合わせてください）
# config_path = "./logs/uudb_csj21/config.json"
# checkpoint_path = "./logs/uudb_csj21/G_3020000.pth"

if not os.path.exists(config_path):
    print(f"ERROR: Config file not found at {config_path}")
elif not os.path.exists(checkpoint_path):
    print(f"ERROR: Checkpoint file not found at {checkpoint_path}")
else:
    hps = utils.get_hparams_from_file(config_path)
    device = "cpu"

    print("Loading model...")
    net_g = SynthesizerTrn(
        len(symbols),
        hps.data.filter_length // 2 + 1,
        hps.train.segment_size // hps.data.hop_length,
        n_speakers=hps.data.n_speakers,
        **hps.model).to(device)
    
    _ = net_g.eval()
    _ = utils.load_checkpoint(checkpoint_path, net_g, None)

    print(f"Original text: {text_to_synthesize}")
    stn_tst = text_to_sequence_custom(text_to_synthesize, hps)
    i = 0 
    
    print(f"\n--- Synthesizing for Speaker {i} (Fixed Ratio OLA / No-Correlation) ---")
    
    with torch.no_grad():
        x_tst = stn_tst.unsqueeze(0).to(device)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        sid = torch.LongTensor([i]).to(device)
            
        # 1. 潜在表現 z の生成
        attn, y_mask, (z, z_p, m_p, logs_p), timings = net_g.infer_z_only(
            x_tst, x_tst_lengths, sid=sid, noise_scale=0.1, noise_scale_w=1.0, length_scale=1.0
        )
        z = z * y_mask
        g = net_g.emb_g(sid).unsqueeze(-1) if net_g.n_speakers > 0 else None

        # 2. パラメータ設定
        z_chunk_size = 10     # z領域でのチャンクサイズ
        z_hop_size = 5        # z領域でのホップサイズ
        
        z_total_frames = z.shape[2]
        full_complex_spec = None
        
        start_time_dec = time.time()
        
        # 倍率保持用変数
        time_ratio = None
        
        # 3. スペクトログラム生成＆結合ループ
        for idx in range(0, z_total_frames, z_hop_size):
            z_end = min(idx + z_chunk_size, z_total_frames)
            z_chunk = z[:, :, idx:z_end]
            
            # デコード
            _, _, spec_chunk, phase_chunk = net_g.dec(z_chunk, g=g)
            
            # 複素スペクトログラム
            complex_chunk = spec_chunk * torch.exp(1j * phase_chunk)
            
            # 現在のチャンクの長さ
            current_spec_len = complex_chunk.shape[-1]
            current_z_len = z_chunk.shape[-1]

            # 初回: 倍率(Ratio)を計算
            if time_ratio is None:
                if current_z_len > 0:
                    time_ratio = current_spec_len / current_z_len
                else:
                    time_ratio = 1.0
                print(f"Time Ratio (Spec / Z) = {time_ratio:.4f}")

            if full_complex_spec is None:
                # 最初のチャンクはそのまま採用
                full_complex_spec = complex_chunk
                print(f"Step {idx}: Init Size {current_spec_len}")
            else:
                # --- 固定計算によるオーバーラップ ---
                
                # 1. 今回「新規に進んだ」zフレーム数に対応する、スペクトログラム上の長さを計算
                #    通常は z_hop_size 分進むが、最後尾付近では短くなる可能性がある
                z_advance = min(z_hop_size, current_z_len) # 今回のzの有効長さではなく、進み幅ベースで考えるべき
                # しかし、ループは z_hop_size 固定で回っているので、基本は z_hop_size
                
                # スペクトログラム上で進むべきフレーム数 (Hop)
                spec_hop_len = int(z_hop_size * time_ratio)
                
                # オーバーラップさせるべき長さ = (現在のチャンク全長) - (進むべき長さ)
                spec_overlap_len = current_spec_len - spec_hop_len
                
                # 安全策: オーバーラップが異常値にならないかチェック
                if spec_overlap_len <= 0:
                    # ホップ量がチャンク長を超えている場合（通常ありえないが念のため）
                    full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
                    print(f"Step {idx}: No overlap append (Hop {spec_hop_len})")
                    continue
                
                # --- クロスフェード (単純なリニア補間) ---
                
                # 前回の末尾 (Fade Out対象)
                prev_tail = full_complex_spec[..., -spec_overlap_len:]
                # 今回の先頭 (Fade In対象)
                curr_head = complex_chunk[..., :spec_overlap_len]
                
                # サイズが一致することを確認（端数処理などでずれる場合への対応）
                actual_overlap = min(prev_tail.shape[-1], curr_head.shape[-1])
                
                if actual_overlap > 0:
                    # アルファ値生成 (0 -> 1)
                    alpha_shape = [1] * (complex_chunk.dim() - 1) + [actual_overlap]
                    alpha = torch.linspace(0.0, 1.0, actual_overlap).to(device).view(*alpha_shape)
                    
                    # 混ぜ合わせ
                    part_prev = prev_tail[..., :actual_overlap] * (1 - alpha)
                    part_curr = curr_head[..., :actual_overlap] * alpha
                    merged_part = part_prev + part_curr
                    
                    # 結合: [前回の確定部分] + [混合部分] + [今回の新規部分]
                    full_complex_spec = torch.cat([
                        full_complex_spec[..., :-actual_overlap], # 前回のオーバーラップ手前まで
                        merged_part,                              # 混合したオーバーラップ部分
                        complex_chunk[..., actual_overlap:]       # 今回の残り（新規）
                    ], dim=-1)
                    
                    # 新規に追加されたフレーム数（ログ用）
                    new_frames = complex_chunk.shape[-1] - actual_overlap
                    print(f"Step {idx:03}: Hop={spec_hop_len} | Overlap={actual_overlap} | New={new_frames}")
                    
                else:
                    # オーバーラップできない場合
                    full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)

            if z_end == z_total_frames:
                break

        # 4. 一括 iSTFT & 合成
        final_spec = torch.abs(full_complex_spec)
        final_phase = torch.angle(full_complex_spec)
        
        stft = TorchSTFT(
            filter_length=net_g.dec.gen_istft_n_fft, 
            hop_length=net_g.dec.gen_istft_hop_size, 
            win_length=net_g.dec.gen_istft_n_fft
        ).to(device)

        if hasattr(net_g.dec, 'subbands') and net_g.dec.subbands > 1:
            b, s, f, t = final_spec.shape
            spec_reshaped = final_spec.view(b * s, f, t)
            phase_reshaped = final_phase.view(b * s, f, t)
            
            y_mb_hat = stft.inverse(spec_reshaped, phase_reshaped)
            y_mb_hat = y_mb_hat.squeeze(1).view(b, s, -1)
            
            if net_g.ms_istft_vits:
                y_mb_hat = F.conv_transpose1d(y_mb_hat, net_g.dec.updown_filter.to(device) * net_g.dec.subbands, stride=net_g.dec.subbands)
                audio_tensor = net_g.dec.multistream_conv_post(y_mb_hat)
            else:
                try:
                    from pqmf import PQMF
                    pqmf = PQMF(device)
                    audio_tensor = pqmf.synthesis(y_mb_hat.unsqueeze(2)) 
                except ImportError:
                     audio_tensor = torch.sum(y_mb_hat, dim=1, keepdim=True)
        else:
            audio_tensor = stft.inverse(final_spec, final_phase)
        
        end_time = time.time()
        audio = audio_tensor[0, 0].data.cpu().float().numpy()

    elapsed_time = end_time - start_time_dec
    audio_duration = len(audio) / hps.data.sampling_rate
    
    print(f"Total Audio duration: {audio_duration:.2f} seconds")
    print(f"Decoding Elapsed time: {elapsed_time:.2f} seconds")
    
    display(Audio(audio, rate=hps.data.sampling_rate, normalize=False))
    print("\nSynthesis complete.")

Loading model...
Mutli-stream iSTFT VITS
Original text: 最近、インターステラーを見たのですけど、すごく面白かったです

--- Synthesizing for Speaker 0 (Fixed Ratio OLA / No-Correlation) ---
Time Ratio (Spec / Z) = 16.1000
Step 0: Init Size 161
Step 005: Hop=80 | Overlap=81 | New=80
Step 010: Hop=80 | Overlap=81 | New=80
Step 015: Hop=80 | Overlap=81 | New=80
Step 020: Hop=80 | Overlap=81 | New=80
Step 025: Hop=80 | Overlap=81 | New=80
Step 030: Hop=80 | Overlap=81 | New=80
Step 035: Hop=80 | Overlap=81 | New=80
Step 040: Hop=80 | Overlap=81 | New=80
Step 045: Hop=80 | Overlap=81 | New=80
Step 050: Hop=80 | Overlap=81 | New=80
Step 055: Hop=80 | Overlap=81 | New=80
Step 060: Hop=80 | Overlap=81 | New=80
Step 065: Hop=80 | Overlap=81 | New=80
Step 070: Hop=80 | Overlap=81 | New=80
Step 075: Hop=80 | Overlap=81 | New=80
Step 080: Hop=80 | Overlap=81 | New=80
Step 085: Hop=80 | Overlap=81 | New=80
Step 090: Hop=80 | Overlap=81 | New=80
Step 095: Hop=80 | Overlap=81 | New=80
Step 100: Hop=80 | Overlap=81 | New=80
Step 105


Synthesis complete.


# 6 細切れデコード　スペクトログラムを接合　overlapあり　相互相関で調整あり

In [11]:
import os
import time
import numpy as np
import torch
import torch.nn.functional as F
import utils
from models import SynthesizerTrn
from text_JP import cleaned_text_to_sequence, symbols
from IPython.display import Audio, display

# ==========================================
# 1. 必要なクラス・関数定義
# ==========================================

class TorchSTFT(torch.nn.Module):
    def __init__(self, filter_length=800, hop_length=200, win_length=800, window='hann'):
        super().__init__()
        self.filter_length = filter_length
        self.hop_length = hop_length
        self.win_length = win_length
        if window == 'hann':
            self.window = torch.hann_window(win_length, periodic=True)
        else:
            self.window = torch.hann_window(win_length, periodic=True)

    def inverse(self, magnitude, phase):
        complex_spec = magnitude * torch.exp(phase * 1j)
        inverse_transform = torch.istft(
            complex_spec,
            self.filter_length, self.hop_length, self.win_length, 
            window=self.window.to(complex_spec.device)
        )
        return inverse_transform.unsqueeze(1)

def find_best_frame_shift(ref_spec, target_spec, search_range=5):
    # 4次元対応
    if ref_spec.dim() == 4:
        b, s, f, t = ref_spec.shape
        ref_spec = ref_spec.reshape(b, s * f, t).contiguous()
        target_spec = target_spec.reshape(b, s * f, t).contiguous()
    
    # 対数振幅で相関をとる
    ref_log = torch.log(ref_spec + 1e-6)
    target_log = torch.log(target_spec + 1e-6)

    ref_log = ref_log - torch.mean(ref_log, dim=-1, keepdim=True)
    target_log = target_log - torch.mean(target_log, dim=-1, keepdim=True)

    pad_target = F.pad(target_log, (search_range, search_range))
    cross_corr = F.conv1d(pad_target, ref_log)
    
    max_idx = torch.argmax(cross_corr)
    shift = max_idx.item() - search_range
    return shift

# ==========================================
# 2. メイン合成プロセス (修正版)
# ==========================================

if not os.path.exists(config_path):
    print(f"ERROR: Config file not found at {config_path}")
elif not os.path.exists(checkpoint_path):
    print(f"ERROR: Checkpoint file not found at {checkpoint_path}")
else:
    hps = utils.get_hparams_from_file(config_path)
    device = "cpu"

    print("Loading model...")
    net_g = SynthesizerTrn(
        len(symbols),
        hps.data.filter_length // 2 + 1,
        hps.train.segment_size // hps.data.hop_length,
        n_speakers=hps.data.n_speakers,
        **hps.model).to(device)
    
    _ = net_g.eval()
    _ = utils.load_checkpoint(checkpoint_path, net_g, None)

    print(f"Original text: {text_to_synthesize}")
    stn_tst = text_to_sequence_custom(text_to_synthesize, hps)
    i = 300
    
    print(f"\n--- Synthesizing for Speaker {i} (Corrected Ratio OLA) ---")
    
    with torch.no_grad():
        x_tst = stn_tst.unsqueeze(0).to(device)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        sid = torch.LongTensor([i]).to(device)
            
        # 1. 潜在表現 z の生成
        attn, y_mask, (z, z_p, m_p, logs_p), timings = net_g.infer_z_only(
            x_tst, x_tst_lengths, sid=sid, noise_scale=0.1, noise_scale_w=1.0, length_scale=1.0
        )
        z = z * y_mask
        g = net_g.emb_g(sid).unsqueeze(-1) if net_g.n_speakers > 0 else None

        # 2. パラメータ設定
        z_chunk_size = 15     # z領域でのチャンクサイズ
        z_hop_size = 5        # z領域でのホップサイズ
        
        search_range = 2      # 探索範囲 (スペクトログラムフレーム単位)
        
        z_total_frames = z.shape[2]
        full_complex_spec = None
        
        start_time_dec = time.time()
        
        # 倍率保持用変数
        time_ratio = None
        
        # 3. スペクトログラム生成＆結合ループ
        for idx in range(0, z_total_frames, z_hop_size):
            z_end = min(idx + z_chunk_size, z_total_frames)
            z_chunk = z[:, :, idx:z_end]
            
            # デコード
            _, _, spec_chunk, phase_chunk = net_g.dec(z_chunk, g=g)
            
            # 複素スペクトログラム
            complex_chunk = spec_chunk * torch.exp(1j * phase_chunk)
            
            # 現在のチャンクの実際の長さ (スペクトログラムフレーム数)
            current_spec_len = complex_chunk.shape[-1]
            current_z_len = z_chunk.shape[-1]

            # 初回のみ倍率を計算
            if time_ratio is None:
                if current_z_len > 0:
                    time_ratio = current_spec_len / current_z_len
                else:
                    time_ratio = 1.0 # fallback
                print(f"Detected Time Ratio (Spec/Z): {time_ratio:.2f}")

            if full_complex_spec is None:
                full_complex_spec = complex_chunk
                print(f"Step {idx}: Init Size {current_spec_len}")
            else:
                # --- 正しいオーバーラップ量の計算 ---
                # 今回「新規に進んだ」zフレーム数に対応するスペクトログラム長を計算
                # idx は z_hop_size ずつ進んでいる
                
                # 直前のステップからの z の進み幅 (通常は z_hop_size だが最後は短いかも)
                # 正確には「今回のチャンクがカバーするz領域」と「前回までのz領域」の重なりを計算すべきだが
                # 単純化のため「期待されるホップ長」を計算する
                
                spec_hop_len = int(z_hop_size * time_ratio)
                
                # オーバーラップ長 = 全長 - ホップ長
                # これが「前回の末尾」と「今回の先頭」で重なるべき長さ
                spec_overlap_len = current_spec_len - spec_hop_len
                
                # オーバーラップがマイナスになる（ホップしすぎ）場合は単に結合
                if spec_overlap_len <= 0:
                    full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
                    print(f"Step {idx}: No overlap (Hop {spec_hop_len} >= Len {current_spec_len})")
                    continue

                # --- 相互相関による位置合わせ ---
                prev_tail = full_complex_spec[..., -spec_overlap_len:]
                curr_head = complex_chunk[..., :spec_overlap_len]
                
                shift = 0
                # サイズチェック
                if prev_tail.shape[-1] == spec_overlap_len and curr_head.shape[-1] == spec_overlap_len:
                    shift = find_best_frame_shift(torch.abs(prev_tail), torch.abs(curr_head), search_range)
                
                # シフト適用
                start_offset = -shift
                start_offset = max(0, min(start_offset, search_range * 2))
                
                aligned_chunk = complex_chunk[..., start_offset:]
                
                # クロスフェード長 (オーバーラップ領域全体にかける)
                cross_len = min(spec_overlap_len, aligned_chunk.shape[-1])
                
                # ログ出力
                new_frames = aligned_chunk.shape[-1] - cross_len
                print(f"Step {idx:03}: RatioOverlap={spec_overlap_len} | Shift={shift:+d} | New={new_frames}")

                if cross_len > 0:
                    alpha_shape = [1] * (aligned_chunk.dim() - 1) + [cross_len]
                    alpha = torch.linspace(0.0, 1.0, cross_len).to(device).view(*alpha_shape)
                    
                    part_prev = full_complex_spec[..., -cross_len:] * (1 - alpha)
                    part_curr = aligned_chunk[..., :cross_len] * alpha
                    merged_part = part_prev + part_curr
                    
                    full_complex_spec = torch.cat([
                        full_complex_spec[..., :-cross_len], 
                        merged_part, 
                        aligned_chunk[..., cross_len:]
                    ], dim=-1)
                else:
                    full_complex_spec = torch.cat([full_complex_spec, aligned_chunk], dim=-1)

            if z_end == z_total_frames:
                break

        # 4. 一括 iSTFT & 合成 (前回と同様)
        final_spec = torch.abs(full_complex_spec)
        final_phase = torch.angle(full_complex_spec)
        
        stft = TorchSTFT(
            filter_length=net_g.dec.gen_istft_n_fft, 
            hop_length=net_g.dec.gen_istft_hop_size, 
            win_length=net_g.dec.gen_istft_n_fft
        ).to(device)

        if hasattr(net_g.dec, 'subbands') and net_g.dec.subbands > 1:
            b, s, f, t = final_spec.shape
            spec_reshaped = final_spec.view(b * s, f, t)
            phase_reshaped = final_phase.view(b * s, f, t)
            
            y_mb_hat = stft.inverse(spec_reshaped, phase_reshaped)
            y_mb_hat = y_mb_hat.squeeze(1).view(b, s, -1)
            
            if net_g.ms_istft_vits:
                y_mb_hat = F.conv_transpose1d(y_mb_hat, net_g.dec.updown_filter.to(device) * net_g.dec.subbands, stride=net_g.dec.subbands)
                audio_tensor = net_g.dec.multistream_conv_post(y_mb_hat)
            else:
                try:
                    from pqmf import PQMF
                    pqmf = PQMF(device)
                    audio_tensor = pqmf.synthesis(y_mb_hat.unsqueeze(2)) 
                except ImportError:
                     audio_tensor = torch.sum(y_mb_hat, dim=1, keepdim=True)
        else:
            audio_tensor = stft.inverse(final_spec, final_phase)
        
        end_time = time.time()
        audio = audio_tensor[0, 0].data.cpu().float().numpy()

    elapsed_time = end_time - start_time_dec
    audio_duration = len(audio) / hps.data.sampling_rate
    
    print(f"Total Audio duration: {audio_duration:.2f} seconds")
    print(f"Decoding Elapsed time: {elapsed_time:.2f} seconds")
    
    display(Audio(audio, rate=hps.data.sampling_rate, normalize=False))
    print("\nSynthesis complete.")

Loading model...
Mutli-stream iSTFT VITS
Original text: 最近、インターステラーを見たのですけど、すごく面白かったです。

--- Synthesizing for Speaker 300 (Corrected Ratio OLA) ---
Detected Time Ratio (Spec/Z): 16.07
Step 0: Init Size 241
Step 005: RatioOverlap=161 | Shift=+2 | New=80
Step 010: RatioOverlap=161 | Shift=+0 | New=80
Step 015: RatioOverlap=161 | Shift=+0 | New=80
Step 020: RatioOverlap=161 | Shift=+2 | New=80
Step 025: RatioOverlap=161 | Shift=-1 | New=79
Step 030: RatioOverlap=161 | Shift=+0 | New=80
Step 035: RatioOverlap=161 | Shift=+0 | New=80
Step 040: RatioOverlap=161 | Shift=+0 | New=80
Step 045: RatioOverlap=161 | Shift=+0 | New=80
Step 050: RatioOverlap=161 | Shift=+0 | New=80
Step 055: RatioOverlap=161 | Shift=+0 | New=80
Step 060: RatioOverlap=161 | Shift=-1 | New=79
Step 065: RatioOverlap=161 | Shift=+0 | New=80
Step 070: RatioOverlap=161 | Shift=+0 | New=80
Step 075: RatioOverlap=161 | Shift=+0 | New=80
Step 080: RatioOverlap=161 | Shift=-1 | New=79
Step 085: RatioOverlap=161 | Shift=-2 | Ne


Synthesis complete.


In [14]:
import os
import time
import torch
import torch.nn.functional as F
import numpy as np
import commons
import utils
from models import SynthesizerTrn
from text_JP import symbols
from scipy.io.wavfile import write

# ==========================================
# 1. 設定
# ==========================================
config_path = "./logs/uudb_csj21/config.json"
checkpoint_path = "./logs/uudb_csj21/G_3020000.pth"
input_txt_path = "./filelists/csj_uudb_test_fine.txt"
output_dir = "output_wavs_batch"

# 生成パラメータ
noise_scale = 0.1
noise_scale_w = 1.0
length_scale = 1.0

# デバイス設定
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ==========================================
# 2. 共通クラス・関数定義
# ==========================================

class TorchSTFT(torch.nn.Module):
    def __init__(self, filter_length=800, hop_length=200, win_length=800, window='hann'):
        super().__init__()
        self.filter_length = filter_length
        self.hop_length = hop_length
        self.win_length = win_length
        # windowのデバイス転送はinverse内で行うためここでは作成のみ
        self.window = torch.hann_window(win_length, periodic=True)

    def inverse(self, magnitude, phase):
        complex_spec = magnitude * torch.exp(phase * 1j)
        # istftは (..., Freq, Time) を期待するため、Batch次元等がある場合は維持される
        inverse_transform = torch.istft(
            complex_spec,
            self.filter_length, self.hop_length, self.win_length, 
            window=self.window.to(complex_spec.device)
        )
        return inverse_transform.unsqueeze(1)

def find_best_frame_shift(ref_spec, target_spec, search_range=5):
    # 4次元(B, S, F, T)対応: 全サブバンド・周波数をまとめて相関をとる
    if ref_spec.dim() == 4:
        b, s, f, t = ref_spec.shape
        ref_spec = ref_spec.reshape(b, s * f, t).contiguous()
        target_spec = target_spec.reshape(b, s * f, t).contiguous()
    
    ref_log = torch.log(ref_spec + 1e-6)
    target_log = torch.log(target_spec + 1e-6)
    ref_log = ref_log - torch.mean(ref_log, dim=-1, keepdim=True)
    target_log = target_log - torch.mean(target_log, dim=-1, keepdim=True)

    pad_target = F.pad(target_log, (search_range, search_range))
    cross_corr = F.conv1d(pad_target, ref_log)
    max_idx = torch.argmax(cross_corr)
    return max_idx.item() - search_range

def get_text_from_phonemes(phonemes, hps):
    symbol_to_id = {s: i for i, s in enumerate(symbols)}
    clean_phonemes = phonemes.replace("[", "").replace("]", "").strip()
    phoneme_list = clean_phonemes.split(" ")
    text_norm = []
    for p in phoneme_list:
        if p in symbol_to_id:
            text_norm.append(symbol_to_id[p])
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    return torch.LongTensor(text_norm)

def istft_finalize(net_g, full_complex_spec):
    """
    Multi-stream iSTFTに対応した波形再構成関数
    full_complex_spec shape: [Batch, Subbands, Freq, Time]
    """
    final_spec = torch.abs(full_complex_spec)
    final_phase = torch.angle(full_complex_spec)
    
    stft = TorchSTFT(
        filter_length=net_g.dec.gen_istft_n_fft, 
        hop_length=net_g.dec.gen_istft_hop_size, 
        win_length=net_g.dec.gen_istft_n_fft
    ).to(device)

    # Multi-stream処理の判定
    if hasattr(net_g.dec, 'subbands') and net_g.dec.subbands > 1:
        # [B, S, F, T] -> [B*S, F, T] に変形してiSTFT
        b, s, f, t = final_spec.shape
        spec_reshaped = final_spec.view(b * s, f, t)
        phase_reshaped = final_phase.view(b * s, f, t)
        
        y_mb_hat = stft.inverse(spec_reshaped, phase_reshaped) # -> [B*S, 1, Time_sub]
        y_mb_hat = y_mb_hat.squeeze(1).view(b, s, -1)          # -> [B, S, Time_sub]

        # 合成フィルタ (Synthesis Filter Bank)
        if net_g.ms_istft_vits:
            # 学習済みアップサンプリングフィルタを使用
            y_mb_hat = F.conv_transpose1d(
                y_mb_hat, 
                net_g.dec.updown_filter.to(device) * net_g.dec.subbands, 
                stride=net_g.dec.subbands
            )
            audio_tensor = net_g.dec.multistream_conv_post(y_mb_hat)
        else:
            # PQMFまたは単純加算 (Fallback)
            try:
                from pqmf import PQMF
                pqmf = PQMF(device)
                audio_tensor = pqmf.synthesis(y_mb_hat.unsqueeze(2)) 
            except ImportError:
                 audio_tensor = torch.sum(y_mb_hat, dim=1, keepdim=True)
    else:
        # 通常のiSTFT (Single stream)
        audio_tensor = stft.inverse(final_spec, final_phase)

    return audio_tensor[0, 0].data.cpu().float().numpy()


# ==========================================
# 3. 各合成条件の関数
# ==========================================

# (1) Normal
def synthesize_cond1(net_g, x_tst, x_tst_lengths, sid):
    # inferメソッドは内部で適切にMulti-stream処理を行うためそのまま使用
    audio = net_g.infer(
        x_tst, x_tst_lengths, sid=sid, 
        noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale
    )[0][0,0].data.cpu().float().numpy()
    return audio

# (2) Audio Chunk
def synthesize_cond2(net_g, x_tst, x_tst_lengths, sid, chunk_size=10):
    attn, y_mask, (z, z_p, m_p, logs_p), timings = net_g.infer_z_only(
        x_tst, x_tst_lengths, sid=sid, noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale
    )
    z = z * y_mask
    g = net_g.emb_g(sid).unsqueeze(-1) if net_g.n_speakers > 0 else None
    
    full_audio_chunks = []
    z_len = z.shape[2]
    # 出力波形の結合が必要
    for step in range(0, z_len, chunk_size):
        z_chunk = z[:, :, step:min(step+chunk_size, z_len)]
        o_chunk, _, _, _ = net_g.dec(z_chunk, g=g)
        full_audio_chunks.append(o_chunk)
    
    return torch.cat(full_audio_chunks, dim=2)[0, 0].data.cpu().float().numpy()

# (3) Spec Fixed Ratio OLA
def synthesize_cond3(net_g, x_tst, x_tst_lengths, sid, z_chunk_size=10, z_hop_size=5):
    attn, y_mask, (z, z_p, m_p, logs_p), timings = net_g.infer_z_only(
        x_tst, x_tst_lengths, sid=sid, noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale
    )
    z = z * y_mask
    g = net_g.emb_g(sid).unsqueeze(-1) if net_g.n_speakers > 0 else None
    
    z_total = z.shape[2]
    full_spec = None
    ratio = None
    
    for idx in range(0, z_total, z_hop_size):
        z_chunk = z[:, :, idx:min(idx+z_chunk_size, z_total)]
        _, _, spec, phase = net_g.dec(z_chunk, g=g)
        comp_chunk = spec * torch.exp(1j * phase)
        
        if ratio is None: 
            ratio = comp_chunk.shape[-1] / z_chunk.shape[-1] if z_chunk.shape[-1] > 0 else 1.0
        
        if full_spec is None:
            full_spec = comp_chunk
        else:
            hop = int(z_hop_size * ratio)
            overlap = comp_chunk.shape[-1] - hop
            if overlap <= 0:
                full_spec = torch.cat([full_spec, comp_chunk], dim=-1)
            else:
                prev, curr = full_spec[..., -overlap:], comp_chunk[..., :overlap]
                ov_len = min(prev.shape[-1], curr.shape[-1])
                if ov_len > 0:
                    alpha = torch.linspace(0.0, 1.0, ov_len).to(device).view(1, 1, 1, ov_len)
                    merged = prev[..., :ov_len]*(1-alpha) + curr[..., :ov_len]*alpha
                    full_spec = torch.cat([full_spec[..., :-ov_len], merged, comp_chunk[..., ov_len:]], dim=-1)
                else:
                    full_spec = torch.cat([full_spec, comp_chunk], dim=-1)
        if idx + z_chunk_size >= z_total: break
            
    return istft_finalize(net_g, full_spec)

# (4) Spec Corrected Ratio OLA
def synthesize_cond4(net_g, x_tst, x_tst_lengths, sid, z_chunk_size=10, z_hop_size=5, search_range=2):
    attn, y_mask, (z, z_p, m_p, logs_p), timings = net_g.infer_z_only(
        x_tst, x_tst_lengths, sid=sid, noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale
    )
    z = z * y_mask
    g = net_g.emb_g(sid).unsqueeze(-1) if net_g.n_speakers > 0 else None
    
    z_total = z.shape[2]
    full_spec = None
    ratio = None
    
    for idx in range(0, z_total, z_hop_size):
        z_chunk = z[:, :, idx:min(idx+z_chunk_size, z_total)]
        _, _, spec, phase = net_g.dec(z_chunk, g=g)
        comp_chunk = spec * torch.exp(1j * phase)
        
        if ratio is None: 
            ratio = comp_chunk.shape[-1] / z_chunk.shape[-1] if z_chunk.shape[-1] > 0 else 1.0
        
        if full_spec is None:
            full_spec = comp_chunk
        else:
            hop = int(z_hop_size * ratio)
            overlap = comp_chunk.shape[-1] - hop
            if overlap <= 0:
                full_spec = torch.cat([full_spec, comp_chunk], dim=-1)
                continue
            
            prev = full_spec[..., -overlap:]
            curr = comp_chunk[..., :overlap]
            
            shift = 0
            # オーバーラップサイズが十分にある場合のみ位置合わせ
            if prev.shape[-1] == overlap and curr.shape[-1] == overlap:
                shift = find_best_frame_shift(torch.abs(prev), torch.abs(curr), search_range)
            
            start_off = max(0, min(-shift, search_range*2))
            aligned = comp_chunk[..., start_off:]
            cross_len = min(overlap, aligned.shape[-1])
            
            if cross_len > 0:
                alpha = torch.linspace(0.0, 1.0, cross_len).to(device).view(1, 1, 1, cross_len)
                merged = full_spec[..., -cross_len:]*(1-alpha) + aligned[..., :cross_len]*alpha
                full_spec = torch.cat([full_spec[..., :-cross_len], merged, aligned[..., cross_len:]], dim=-1)
            else:
                full_spec = torch.cat([full_spec, aligned], dim=-1)
        if idx + z_chunk_size >= z_total: break

    return istft_finalize(net_g, full_spec)


# ==========================================
# 4. メインループ
# ==========================================
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory created: {output_dir}")

hps = utils.get_hparams_from_file(config_path)
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    n_speakers=hps.data.n_speakers,
    **hps.model).to(device)
_ = net_g.eval()
utils.load_checkpoint(checkpoint_path, net_g, None)
print("Model loaded.")

with open(input_txt_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"Start processing {len(lines)} lines...")

for i, line in enumerate(lines):
    line = line.strip()
    if not line: continue
    parts = line.split("|")
    if len(parts) < 3: continue
    
    original_fname = os.path.basename(parts[0]).replace(".wav", "")
    spk_id = int(parts[1])
    phonemes = parts[2]
    
    stn_tst = get_text_from_phonemes(phonemes, hps)
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        sid = torch.LongTensor([spk_id]).to(device)

        try:
            audio = synthesize_cond1(net_g, x_tst, x_tst_lengths, sid)
            write(os.path.join(output_dir, f"{original_fname}_cond1.wav"), hps.data.sampling_rate, audio)
            
            audio = synthesize_cond2(net_g, x_tst, x_tst_lengths, sid)
            write(os.path.join(output_dir, f"{original_fname}_cond2.wav"), hps.data.sampling_rate, audio)
            
            audio = synthesize_cond3(net_g, x_tst, x_tst_lengths, sid)
            write(os.path.join(output_dir, f"{original_fname}_cond3.wav"), hps.data.sampling_rate, audio)
            
            audio = synthesize_cond4(net_g, x_tst, x_tst_lengths, sid)
            write(os.path.join(output_dir, f"{original_fname}_cond4.wav"), hps.data.sampling_rate, audio)
            
            print(f"[{i+1}/{len(lines)}] Saved: {original_fname}")
        except Exception as e:
            print(f"Error processing {original_fname}: {e}")
            import traceback
            traceback.print_exc()

print("All tasks finished.")

Using device: cuda
Output directory created: output_wavs_batch
Mutli-stream iSTFT VITS
Model loaded.
Start processing 16 lines...
[1/16] Saved: FJK_C051_118
[2/16] Saved: FJK_C051_170
[3/16] Saved: FKC_C031_002
[4/16] Saved: FMS_C051_072
[5/16] Saved: FMT_C041_134
[6/16] Saved: FMT_C041_259
[7/16] Saved: FTH_C004_044
[8/16] Saved: FTH_C005_152
[9/16] Saved: FTS_C002_107
[10/16] Saved: FTS_C002_175
[11/16] Saved: FTS_C004_126
[12/16] Saved: FTS_C006_050
[13/16] Saved: FTS_C007_137
[14/16] Saved: FUE_C033_134
[15/16] Saved: FYH_C042_090
[16/16] Saved: FYH_C043_064
All tasks finished.


# 共通クラス　設定

In [4]:
import os
import time
import torch
import torch.nn.functional as F
import numpy as np
import commons
import utils
from models import SynthesizerTrn
from text_JP import symbols
from scipy.io.wavfile import write

# ==========================================
# 1. 設定
# ==========================================
config_path = "./logs/uudb_csj21/config.json"
checkpoint_path = "./logs/uudb_csj21/G_3020000.pth"
input_txt_path = "./filelists/csj_uudb_test_fine.txt"
output_dir = "output_wavs_batch"

# 生成パラメータ
noise_scale = 0.1
noise_scale_w = 1.0
length_scale = 1.0

# デバイス設定
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ==========================================
# 2. 共通クラス・関数定義
# ==========================================

class TorchSTFT(torch.nn.Module):
    def __init__(self, filter_length=800, hop_length=200, win_length=800, window='hann'):
        super().__init__()
        self.filter_length = filter_length
        self.hop_length = hop_length
        self.win_length = win_length
        # windowのデバイス転送はinverse内で行うためここでは作成のみ
        self.window = torch.hann_window(win_length, periodic=True)

    def inverse(self, magnitude, phase):
        complex_spec = magnitude * torch.exp(phase * 1j)
        # istftは (..., Freq, Time) を期待するため、Batch次元等がある場合は維持される
        inverse_transform = torch.istft(
            complex_spec,
            self.filter_length, self.hop_length, self.win_length, 
            window=self.window.to(complex_spec.device)
        )
        return inverse_transform.unsqueeze(1)

def find_best_frame_shift(ref_spec, target_spec, search_range=5):
    # 4次元(B, S, F, T)対応: 全サブバンド・周波数をまとめて相関をとる
    if ref_spec.dim() == 4:
        b, s, f, t = ref_spec.shape
        ref_spec = ref_spec.reshape(b, s * f, t).contiguous()
        target_spec = target_spec.reshape(b, s * f, t).contiguous()
    
    ref_log = torch.log(ref_spec + 1e-6)
    target_log = torch.log(target_spec + 1e-6)
    ref_log = ref_log - torch.mean(ref_log, dim=-1, keepdim=True)
    target_log = target_log - torch.mean(target_log, dim=-1, keepdim=True)

    pad_target = F.pad(target_log, (search_range, search_range))
    cross_corr = F.conv1d(pad_target, ref_log)
    max_idx = torch.argmax(cross_corr)
    return max_idx.item() - search_range

def get_text_from_phonemes(phonemes, hps):
    symbol_to_id = {s: i for i, s in enumerate(symbols)}
    clean_phonemes = phonemes.replace("[", "").replace("]", "").strip()
    phoneme_list = clean_phonemes.split(" ")
    text_norm = []
    for p in phoneme_list:
        if p in symbol_to_id:
            text_norm.append(symbol_to_id[p])
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    return torch.LongTensor(text_norm)

def istft_finalize(net_g, full_complex_spec):
    """
    Multi-stream iSTFTに対応した波形再構成関数
    full_complex_spec shape: [Batch, Subbands, Freq, Time]
    """
    device = full_complex_spec.device
    final_spec = torch.abs(full_complex_spec)
    final_phase = torch.angle(full_complex_spec)
    
    stft = TorchSTFT(
        filter_length=net_g.dec.gen_istft_n_fft, 
        hop_length=net_g.dec.gen_istft_hop_size, 
        win_length=net_g.dec.gen_istft_n_fft
    ).to(device)

    # Multi-stream処理の判定
    if hasattr(net_g.dec, 'subbands') and net_g.dec.subbands > 1:
        # [B, S, F, T] -> [B*S, F, T] に変形してiSTFT
        b, s, f, t = final_spec.shape
        spec_reshaped = final_spec.view(b * s, f, t)
        phase_reshaped = final_phase.view(b * s, f, t)
        
        y_mb_hat = stft.inverse(spec_reshaped, phase_reshaped) # -> [B*S, 1, Time_sub]
        y_mb_hat = y_mb_hat.squeeze(1).view(b, s, -1)          # -> [B, S, Time_sub]

        # 合成フィルタ (Synthesis Filter Bank)
        if net_g.ms_istft_vits:
            # 学習済みアップサンプリングフィルタを使用
            y_mb_hat = F.conv_transpose1d(
                y_mb_hat, 
                net_g.dec.updown_filter.to(device) * net_g.dec.subbands, 
                stride=net_g.dec.subbands
            )
            audio_tensor = net_g.dec.multistream_conv_post(y_mb_hat)
        else:
            # PQMFまたは単純加算 (Fallback)
            try:
                from pqmf import PQMF
                pqmf = PQMF(device)
                audio_tensor = pqmf.synthesis(y_mb_hat.unsqueeze(2)) 
            except ImportError:
                 audio_tensor = torch.sum(y_mb_hat, dim=1, keepdim=True)
    else:
        # 通常のiSTFT (Single stream)
        audio_tensor = stft.inverse(final_spec, final_phase)

    return audio_tensor[0, 0].data.cpu().float().numpy()

Using device: cuda


## new cond1

モーラ単位で分割g2p

In [5]:
import re
import numpy as np
import torch
import pyopenjtalk
from text_JP.phonemize import Phonemizer  # ユーザー環境に合わせてインポート

# ==========================================
# 1. テキスト解析・モーラ分割ロジック
# ==========================================

def get_mora_chunks_from_text(text, phonemizer):
    """
    日本語テキストを受け取り、以下のルールで分割された「音素文字列のリスト」を返す
      - [ ... ] や {cough}、句読点: そのまま1つの塊として処理
      - 通常テキスト: モーラ単位 (例: 'カ', 'ッ', 'パ') に分割して個別に音素化
    """
    # ユーザー指定の正規表現でパース
    # {cough}, <cough>, [content], 句読点 を分離
    parts = re.split(r'({cough}|<cough>|\[.*?\]|[、。])', text)
    
    mora_phoneme_chunks = []
    
    # モーラ抽出用正規表現
    # カタカナ(小文字伴う場合あり) | 長音 | 促音 | 撥音
    mora_pattern = re.compile(r'([ァ-ヴ](?:[ャュョァィゥェォ]?)?|ー|ッ|ン)')

    for part in parts:
        if not part or part.isspace():
            continue
            
        # --- A. ブラケット [ ... ] ---
        if part.startswith('[') and part.endswith(']') and len(part) > 2:
            content = part[1:-1]
            if not content:
                mora_phoneme_chunks.append('[ ]')
            else:
                # 中身をカナ変換 -> 音素化してブラケットで包む
                kana_content = pyopenjtalk.g2p(content, kana=True).replace('ヲ', 'オ')
                phoneme_content = phonemizer(kana_content)
                mora_phoneme_chunks.append(f'[ {phoneme_content} ]')
            continue
        
        # --- B. 特殊タグ {cough} ---
        if part == '{cough}' or part == '<cough>':
            mora_phoneme_chunks.append('<cough>')
            continue
            
        # --- C. 句読点 -> sp ---
        if part in '、。':
            mora_phoneme_chunks.append('sp')
            continue
            
        # --- D. 通常テキスト -> モーラ分割 ---
        # 1. 全体をカナに変換
        kana_full = pyopenjtalk.g2p(part, kana=True).replace('ヲ', 'オ')
        
        # 2. モーラ単位にリスト化 (例: "シ","ャ","シ","ン")
        moras = mora_pattern.findall(kana_full)
        
        # 3. 各モーラを個別に音素化してリスト追加
        for m in moras:
            p = phonemizer(m)
            # 空白除去して追加
            p = p.strip()
            if p:
                mora_phoneme_chunks.append(p)
            
    return mora_phoneme_chunks

# ==========================================
# 2. Cond 1: モーラ単位単純接続 (Cond 1)
# ==========================================

def synthesize_cond1_auto_mora(net_g, raw_text, sid, hps, phonemizer, mora_chunk_size=1):
    """
    Cond 1: 単純接続 (モーラ単位でテキスト分割 -> 個別生成 -> 波形接続)
    引数 mora_chunk_size: 一度に生成するモーラの数 (デフォルト1)
    """
    # モデルのデバイスを取得
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # 1. テキストをモーラ単位の音素列リストに変換
    # 例: ['d e', 'm o', '[ n a N k a ]', ...]
    original_chunks = get_mora_chunks_from_text(raw_text, phonemizer)
    
    # 2. mora_chunk_size ずつ結合して、処理単位を大きくする
    merged_chunks = []
    temp_chunk = []
    
    for ph in original_chunks:
        temp_chunk.append(ph)
        if len(temp_chunk) >= mora_chunk_size:
            # 音素文字列をスペースで結合
            merged_chunks.append(" ".join(temp_chunk))
            temp_chunk = []
            
    # 端数があれば追加
    if temp_chunk:
        merged_chunks.append(" ".join(temp_chunk))
    
    audio_segments = []
    
    # 3. 各チャンクを個別に生成
    for ph in merged_chunks:
        # 音素列をID列に変換
        stn_tst = get_text_from_phonemes(ph, hps)
        
        with torch.no_grad():
            x_tst = stn_tst.to(device).unsqueeze(0)
            x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
            
            # Non-streaming生成 (コンテキスト切断)
            audio = net_g.infer(
                x_tst, x_tst_lengths, sid=sid, 
                noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale
            )[0][0,0].data.cpu().float().numpy()
            
        audio_segments.append(audio)
    
    # 4. 単純結合
    if not audio_segments:
        return np.array([])
        
    full_audio = np.concatenate(audio_segments)
    return full_audio


In [6]:
def get_z_and_phoneme_durations(net_g, x_tst, x_tst_lengths, sid, noise_scale, noise_scale_w, length_scale):
    """
    修正版 (行列積の順序修正): テキスト全体から一括で潜在表現 z と、各音素の継続長を取得する
    """
    # 1. Text Encoder
    # x: [B, C, T_phoneme]
    # m_p: [B, C, T_phoneme], logs_p: [B, C, T_phoneme]
    x, m_p, logs_p, x_mask = net_g.enc_p(x_tst, x_tst_lengths)
    
    # 2. Speaker Embedding
    if net_g.n_speakers > 0:
        g = net_g.emb_g(sid).unsqueeze(-1) # [b, h, 1]
    else:
        g = None

    # 3. Duration Predictor (w_ceil を取得)
    logw = net_g.dp(x, x_mask, g=g)
    w = torch.exp(logw) * x_mask * length_scale
    w_ceil = torch.ceil(w) # [B, 1, T_phoneme]
    
    # 4. Expand (手動アライメントマスク作成)
    B, _, T_phoneme = w_ceil.shape
    T_frame = int(torch.sum(w_ceil).item())
    
    # attn_mask: [B, T_phoneme, T_frame]
    # 「音素iは、フレームjからkまでを担当する」というマスクを作る
    attn_mask = torch.zeros(B, T_phoneme, T_frame).to(x.device)
    
    # バッチサイズ1前提 (推論用)
    w_ceil_flat = w_ceil.squeeze() # [T_phoneme]
    current_frame = 0
    
    for i, dur in enumerate(w_ceil_flat):
        d = int(dur.item())
        if d > 0:
            # 音素 i に対応するフレーム範囲を 1.0 にする
            attn_mask[0, i, current_frame : current_frame + d] = 1.0
            current_frame += d
    
    # Expand mean & variance
    # m_p: [B, Channels, T_phoneme]
    # attn_mask: [B, T_phoneme, T_frame]
    # matmul -> [B, Channels, T_frame]
    m_p = torch.matmul(m_p, attn_mask)
    logs_p = torch.matmul(logs_p, attn_mask)

    # 5. Flow (Reverse) -> z を生成
    # y_mask (フレーム側のマスク) を作成: 全フレーム有効なので全て1
    y_mask = torch.ones(B, 1, T_frame).to(x.device)
    
    z_p = m_p + torch.randn_like(m_p, dtype=torch.float) * torch.exp(logs_p) * noise_scale
    z = net_g.flow(z_p, y_mask, g=g, reverse=True)
    
    return z, w_ceil, g

def synthesize_cond2_auto_mora(net_g, raw_text, sid, hps, phonemizer, mora_chunk_size=1):
    """
    Cond 2: スペクトログラム単純接続 (デコーダのみコンテキスト分断)
    1. テキスト全体 -> 全体z生成 (Full Context)
    2. zをモーラ単位の長さに分割
    3. 個別にデコード -> スペクトログラム単純結合 -> iSTFT
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # --- A. テキスト解析 ---
    chunk_phonemes_list = get_mora_chunks_from_text(raw_text, phonemizer)
    
    # 各モーラの音素数をカウント
    mora_phoneme_counts = []
    all_phoneme_ids = []
    
    for ph_str in chunk_phonemes_list:
        ids = get_text_from_phonemes(ph_str, hps)
        mora_phoneme_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids:
        return np.array([])

    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        # --- B. 全体エンコード ---
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, 
            noise_scale, noise_scale_w, length_scale
        )
        w_ceil_flat = w_ceil.squeeze() 

        # --- C. まとめて切り出し & 個別デコード ---
        full_complex_spec = None
        
        current_ph_idx = 0     # 現在の音素インデックス
        current_z_frame = 0    # 現在のzフレーム位置
        
        # mora_phoneme_counts を mora_chunk_size ずつ処理
        num_chunks = len(mora_phoneme_counts)
        
        for i in range(0, num_chunks, mora_chunk_size):
            # 今回処理するモーラ群の音素数を取得 (例: 2モーラ分なら [2, 3] -> 5音素)
            chunk_counts = mora_phoneme_counts[i : i + mora_chunk_size]
            total_phonemes_in_chunk = sum(chunk_counts)
            
            # このチャンクに対応する継続長を取得して合計
            chunk_durations = w_ceil_flat[current_ph_idx : current_ph_idx + total_phonemes_in_chunk]
            chunk_z_len = int(torch.sum(chunk_durations).item())
            
            # z をスライス
            z_end_frame = current_z_frame + chunk_z_len
            if z_end_frame > z.shape[2]:
                z_end_frame = z.shape[2]
            
            z_chunk = z[:, :, current_z_frame : z_end_frame]
            
            # デコード (ここがセグメント単位)
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                # 単純結合
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
            
            # ポインタ更新
            current_ph_idx += total_phonemes_in_chunk
            current_z_frame = z_end_frame
            
            if current_z_frame >= z.shape[2]:
                break

        # --- D. iSTFT ---
        if full_complex_spec is None:
            return np.array([])
            
        audio = istft_finalize(net_g, full_complex_spec)
        return audio

In [7]:
def synthesize_cond3_auto_mora(net_g, raw_text, sid, hps, phonemizer, z_overlap_frames=5, search_range=2, mora_chunk_size=1):
    """
    Cond 3: 相互相関スペクトログラムOLA (全コンテキストエンコード + モーラ分割デコード)
    
    1. テキスト全体 -> 全体z生成 (Full Context)
    2. zをモーラ単位に分割（ただし隣接区間でオーバーラップを持たせる）
    3. 個別にデコード -> 相互相関で位置合わせしてクロスフェード -> iSTFT
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # --- A. テキスト解析 ---
    chunk_phonemes_list = get_mora_chunks_from_text(raw_text, phonemizer)
    
    mora_phoneme_counts = []
    all_phoneme_ids = []
    for ph_str in chunk_phonemes_list:
        ids = get_text_from_phonemes(ph_str, hps)
        mora_phoneme_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids:
        return np.array([])

    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        # --- B. 全体エンコード ---
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, 
            noise_scale, noise_scale_w, length_scale
        )
        w_ceil_flat = w_ceil.squeeze()

        # --- C. チャンク単位で分割デコード & OLA ---
        full_complex_spec = None
        prev_raw_tail = None
        
        current_ph_idx = 0
        current_z_frame = 0
        ratio = None 
        
        num_chunks = len(mora_phoneme_counts)
        
        # mora_chunk_size ずつまとめてループ
        for i in range(0, num_chunks, mora_chunk_size):
            chunk_counts = mora_phoneme_counts[i : i + mora_chunk_size]
            total_phonemes_in_chunk = sum(chunk_counts)
            
            # 1. このチャンク(複数モーラ)の正味の長さ
            chunk_durations = w_ceil_flat[current_ph_idx : current_ph_idx + total_phonemes_in_chunk]
            chunk_z_len = int(torch.sum(chunk_durations).item())
            
            # 2. 切り出し範囲 (Overlap込み)
            z_end_nominal = current_z_frame + chunk_z_len
            z_end_decode = z_end_nominal + z_overlap_frames
            
            if z_end_decode > z.shape[2]:
                z_end_decode = z.shape[2]
            
            # z 切り出し
            z_chunk = z[:, :, current_z_frame : z_end_decode]
            
            # デコード
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                if ratio is None:
                    ratio = complex_chunk.shape[-1] / z_chunk.shape[-1] if z_chunk.shape[-1] > 0 else 1.0
                
                # --- OLA処理 (前回と同じロジック) ---
                actual_z_overlap = max(0, z_chunk.shape[-1] - chunk_z_len)
                spec_overlap_len = int(actual_z_overlap * ratio)
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    if prev_raw_tail is None: 
                        prev_ref = full_complex_spec[..., -spec_overlap_len:]
                    else:
                        prev_ref = prev_raw_tail
                    
                    curr_ref = complex_chunk[..., :spec_overlap_len]
                    valid_overlap = min(prev_ref.shape[-1], curr_ref.shape[-1])
                    
                    if valid_overlap > 0:
                        shift = 0
                        if valid_overlap > search_range * 2:
                            shift = find_best_frame_shift(
                                torch.abs(prev_ref[..., :valid_overlap]), 
                                torch.abs(curr_ref[..., :valid_overlap]), 
                                search_range
                            )
                        
                        start_off = max(0, min(-shift, search_range * 2))
                        aligned = complex_chunk[..., start_off:]
                        cross_len = min(valid_overlap, aligned.shape[-1])
                        
                        if cross_len > 0:
                            alpha = torch.linspace(0.0, 1.0, cross_len).to(device).view(1, 1, 1, cross_len)
                            merged = prev_ref[..., :cross_len] * (1 - alpha) + aligned[..., :cross_len] * alpha
                            
                            full_complex_spec = torch.cat([
                                full_complex_spec[..., :-cross_len], 
                                merged, 
                                aligned[..., cross_len:]
                            ], dim=-1)
                        else:
                            full_complex_spec = torch.cat([full_complex_spec, aligned], dim=-1)
                    else:
                        full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
                
                # 末尾保存
                next_expected_overlap = int(z_overlap_frames * ratio)
                if complex_chunk.shape[-1] >= next_expected_overlap:
                    prev_raw_tail = complex_chunk[..., -next_expected_overlap:]
                else:
                    prev_raw_tail = complex_chunk

            # ポインタ更新
            current_ph_idx += total_phonemes_in_chunk
            current_z_frame = z_end_nominal # 正味分だけ進める
            
            if current_z_frame >= z.shape[2]:
                break

        # --- D. iSTFT ---
        if full_complex_spec is None:
            return np.array([])
            
        audio = istft_finalize(net_g, full_complex_spec)
        return audio

In [12]:

# ==========================================
# 実行準備
# ==========================================
# Phonemizerのインスタンス化 (ループの外で1回だけ行う)
phonemizer = Phonemizer()

# (以下、メインループ内での呼び出し例)
# ...
raw_japanese_text = "最近、インターステラーを見たのですけど、すごく面白かったです。" #(original_transcriptions_summary.txt等から取得)
#x_tst = stn_tst.to(device).unsqueeze(0)
#x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
sid = torch.LongTensor([374]).to(device)
# ...
audio_cond1 = synthesize_cond1_auto_mora(net_g, raw_japanese_text, sid, hps, phonemizer, 1)
audio_cond2 = synthesize_cond2_auto_mora(net_g, raw_japanese_text, sid, hps, phonemizer, 1)
audio_cond3 = synthesize_cond3_auto_mora(net_g, raw_japanese_text, sid, hps, phonemizer,1,2, 1)
# write(..., audio_cond1)
print("条件１")
display(Audio(audio_cond1, rate=hps.data.sampling_rate, normalize=False))
print("条件２")
display(Audio(audio_cond2, rate=hps.data.sampling_rate, normalize=False))
print("条件３")
display(Audio(audio_cond3, rate=hps.data.sampling_rate, normalize=False))


条件１


条件２


条件３


In [26]:
import time

# ==========================================
# 0. 計測用ヘルパー関数
# ==========================================
def measure_and_print(label, func, *args):
    """
    関数を実行し、所要時間・音声長・RTFを計測して表示する
    """
    # GPUの処理待ちをリセット
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start_time = time.time()
    
    # 関数の実行
    audio = func(*args)
    
    # GPU処理完了まで待機
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    elapsed_time = time.time() - start_time
    
    # 音声の長さを計算 (秒)
    if len(audio) > 0:
        duration = len(audio) / hps.data.sampling_rate
        rtf = elapsed_time / duration
    else:
        duration = 0
        rtf = 0

    print(f"--- {label} ---")
    print(f"Audio duration: {duration:.2f} seconds")
    print(f"Elapsed time: {elapsed_time:.4f} seconds")
    print(f"Real Time Factor (RTF): {rtf:.4f}")
    print("-" * 30)
    
    return audio

def japanese_cleaner_revised(text):
    parts = re.split(r'({cough}|<cough>|\[.*?\]|[、。])', text)
    phoneme_parts = []
    phonemizer = Phonemizer()
    for part in parts:
        if not part or part.isspace():
            continue
        if part.startswith('[') and part.endswith(']') and len(part) > 2:
            content = part[1:-1]
            if not content:
                phoneme_parts.append('[ ]')
            else:
                kana_content = pyopenjtalk.g2p(content, kana=True).replace('ヲ', 'オ')
                phoneme_content = phonemizer(kana_content)
                phoneme_parts.append(f'[ {phoneme_content} ]')
            continue
        if part == '{cough}' or part == '<cough>':
            phoneme_parts.append('<cough>')
            continue
        if part in '、。':
            phoneme_parts.append('sp')
            continue
        kana = pyopenjtalk.g2p(part, kana=True).replace('ヲ', 'オ')
        phonemes = phonemizer(kana)
        phoneme_parts.append(phonemes)
    final_text = ' '.join(phoneme_parts)
    return re.sub(r'\s+', ' ', final_text).strip()

def text_to_sequence_custom(text, hps):
    phonemized_text = japanese_cleaner_revised(text)
    stn_tst = cleaned_text_to_sequence(phonemized_text)
    if hps.data.add_blank:
        stn_tst = commons.intersperse(stn_tst, 0)
    return torch.LongTensor(stn_tst)

# ==========================================
# 比較用: Cond 4 (全体一括生成 / Baseline)
# ==========================================
def synthesize_cond4_full(net_g, raw_text, sid, hps, phonemizer):
    """
    Cond 4: 全体一括生成 (Baseline)
    分割せず、テキスト全体をそのまま生成する
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)
    
    # G2P (全体)
    # 既存のロジックを使って音素化 (特殊記号などは考慮しつつ、分割はしない)
    # ここでは簡易的に結合したものを使用、または phonemizer で全体を一括処理
    # 文脈を維持するため、全体を1つのシーケンスとする
    
    # 簡易実装: 一度 chunk に分けたものを結合して再構成
    # (厳密には original_transcriptions_summary のテキストを直接 g2p するのがベスト)
    
    # ここでは既存の text_to_phoneme 相当のことを行う
    #import pyopenjtalk
    #kana = pyopenjtalk.g2p(raw_text, kana=True).replace('ヲ', 'オ')
    #phonemes = phonemizer(kana)
    stn_tst = text_to_sequence_custom(raw_text, hps)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        audio = net_g.infer(
            x_tst, x_tst_lengths, sid=sid, 
            noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale
        )[0][0,0].data.cpu().float().numpy()
        
    return audio


In [27]:
import os

# ==========================================
# 1. 日本語テキスト (書き起こし) の定義
# ==========================================
raw_input_text = """
uudb/tts1/data/test/wav/FJK_C051_118.wav
  Original: でも[なんかこう] そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた

uudb/tts1/data/test/wav/FJK_C051_170.wav
  Original: やっぱ、やっぱディーが最後かな

uudb/tts1/data/test/wav/FKC_C031_002.wav
  Original: [うんとね] 多分あたし一番持ってる気がするって感じ

uudb/tts1/data/test/wav/FMS_C051_072.wav
  Original: そうだよね、うんうん

uudb/tts1/data/test/wav/FMT_C041_134.wav
  Original: 何だそりゃ

uudb/tts1/data/test/wav/FMT_C041_259.wav
  Original: ディーはエーより後ってことだよね

uudb/tts1/data/test/wav/FTH_C004_044.wav
  Original: [えっとね] じゃあシー、行くね

uudb/tts1/data/test/wav/FTH_C005_152.wav
  Original: [あ] じゃあ、最初は、エーの[その]、交番の外にサラリーマンがいて

uudb/tts1/data/test/wav/FTS_C002_107.wav
  Original: [あ] ちゃんと入ってないんだ、[あー]

uudb/tts1/data/test/wav/FTS_C002_175.wav
  Original: [あー]、オッケーオッケー

uudb/tts1/data/test/wav/FTS_C004_126.wav
  Original: そうね、えっみたいな感じだよね

uudb/tts1/data/test/wav/FTS_C006_050.wav
  Original: なるほど、わかりました

uudb/tts1/data/test/wav/FTS_C007_137.wav
  Original: [あー]、見てるんだ

uudb/tts1/data/test/wav/FUE_C033_134.wav
  Original: [え]、わかんなくなってきた

uudb/tts1/data/test/wav/FYH_C042_090.wav
  Original: この、お母さんがすごい、あんまり上品でない食べ方をしてるのね

uudb/tts1/data/test/wav/FYH_C043_064.wav
  Original: はい、いいですか
"""

# ==========================================
# 2. 話者ID (SID) の参照データ定義
# ==========================================
sid_reference_data = """
uudb/tts1/data/test/wav/FJK_C051_118.wav|368|d e m o [ n a N k a k o: ] s o: s u r u t o sp m a t a m a t a a Q t a k a k u n a Q t a Q t e k o n o n a g a s i m a k a N t o k u g a y u Q t e r u N d a y o n e m a t a
uudb/tts1/data/test/wav/FJK_C051_170.wav|368|y a Q p a sp y a Q p a d i: g a s a i g o k a n a
uudb/tts1/data/test/wav/FKC_C031_002.wav|369|[ u N t o n e ] t a b u N a t a s i i t i b a N m o Q t e r u k i g a s u r u Q t e k a N z i
uudb/tts1/data/test/wav/FMS_C051_072.wav|370|s o: d a y o n e sp u N u N
uudb/tts1/data/test/wav/FMT_C041_134.wav|371|n a N d a s o ry a
uudb/tts1/data/test/wav/FMT_C041_259.wav|371|d i: w a e: y o r i a t o Q t e k o t o d a y o n e
uudb/tts1/data/test/wav/FTH_C004_044.wav|375|[ e Q t o n e ] zy a: s i: sp i k u n e
uudb/tts1/data/test/wav/FTH_C005_152.wav|375|[ a ] zy a: sp s a i sy o w a sp e: n o [ s o n o ] sp k o: b a N n o s o t o n i s a r a r i: m a N g a i t e
uudb/tts1/data/test/wav/FTS_C002_107.wav|376|[ a ] ch a N t o h a i Q t e n a i N d a sp [ a: ]
uudb/tts1/data/test/wav/FTS_C002_175.wav|376|[ a: ] sp o Q k e: o Q k e:
uudb/tts1/data/test/wav/FTS_C004_126.wav|376|s o: n e sp e Q m i t a i n a k a N z i d a y o n e
uudb/tts1/data/test/wav/FTS_C006_050.wav|376|n a r u h o d o sp w a k a r i m a s i t a
uudb/tts1/data/test/wav/FTS_C007_137.wav|376|[ a: ] sp m i t e r u N d a
uudb/tts1/data/test/wav/FUE_C033_134.wav|378|[ e ] sp w a k a N n a k u n a Q t e k i t a
uudb/tts1/data/test/wav/FYH_C042_090.wav|379|k o n o sp o k a: s a N g a s u g o i sp a N m a r i zy o: h i N d e n a i t a b e k a t a o s i t e r u n o n e
uudb/tts1/data/test/wav/FYH_C043_064.wav|379|h a i sp i: d e s u k a
"""

# ==========================================
# 3. データの紐付けとリスト生成処理
# ==========================================

def prepare_execution_list(raw_text, sid_data):
    # 1. SIDマップの作成 (ファイル名 -> SID)
    filename_to_sid = {}
    for line in sid_data.strip().split('\n'):
        line = line.strip()
        if not line: continue
        parts = line.split('|')
        if len(parts) >= 2:
            path = parts[0]
            sid = parts[1]
            fname = os.path.basename(path)
            filename_to_sid[fname] = int(sid)

    print(f"Loaded SID mapping for {len(filename_to_sid)} files.")

    # 2. テキストデータのパースと結合
    execution_lines = []
    current_wav = None
    
    for line in raw_text.strip().split('\n'):
        line = line.strip()
        if not line: continue
        
        if line.endswith('.wav'):
            current_wav = line
        elif line.startswith('Original:'):
            if current_wav:
                text = line.replace('Original:', '').strip()
                fname = os.path.basename(current_wav)
                
                # SIDの引き当て
                if fname in filename_to_sid:
                    sid_val = filename_to_sid[fname]
                    # フォーマット: パス|SID|テキスト
                    formatted_line = f"{current_wav}|{sid_val}|{text}"
                    execution_lines.append(formatted_line)
                else:
                    print(f"Warning: No SID found for {fname}, skipping.")
                
                current_wav = None

    return execution_lines

# リストの生成
lines = prepare_execution_list(raw_input_text, sid_reference_data)

print(f"Prepared {len(lines)} tasks.")
# 確認用出力 (最初の1件)
if lines:
    print("Sample task:", lines[0])

# ==========================================
# 4. メイン実行ループ
# ==========================================
# (以前のコードのループ部分をそのまま使用します)

print("Starting measurement loop...")

# チャンクサイズ設定 (Cond 2, 3用)
mora_chunk = 1

for i, line in enumerate(lines):
    line = line.strip()
    if not line: continue
    parts = line.split("|")
    
    # 読み込み
    file_path = parts[0]
    spk_id_val = int(parts[1])
    raw_text_jp = parts[2]
    
    filename = os.path.basename(file_path).replace(".wav", "")
    sid = torch.LongTensor([spk_id_val]).to(device)
    
    print(f"\nProcessing: {filename} (SID: {spk_id_val})")
    print(f"Text: {raw_text_jp}")

    try:
        # Cond 1: モーラ単位単純接続
        audio_c1 = measure_and_print(
            "Cond 1 (Naive Mora Concat)", 
            synthesize_cond1_auto_mora, 
            net_g, raw_text_jp, sid, hps, phonemizer, mora_chunk
        )
        write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)

        # Cond 2: スペクトログラム単純接続
        audio_c2 = measure_and_print(
            f"Cond 2 (Spec Naive Concat, chunk={mora_chunk})", 
            synthesize_cond2_auto_mora, 
            net_g, raw_text_jp, sid, hps, phonemizer, mora_chunk
        )
        write(os.path.join(output_dir, f"{filename}_cond2.wav"), hps.data.sampling_rate, audio_c2)

        # Cond 3: 相互相関OLA
        audio_c3 = measure_and_print(
            f"Cond 3 (Correlation OLA, chunk={mora_chunk})", 
            synthesize_cond3_auto_mora, 
            net_g, raw_text_jp, sid, hps, phonemizer, 
            1, 2, mora_chunk  # overlap=5, search=2
        )
        write(os.path.join(output_dir, f"{filename}_cond3.wav"), hps.data.sampling_rate, audio_c3)

        # Cond 4: 全体一括 (Baseline)
        audio_c4 = measure_and_print(
            "Cond 4 (Full Context Baseline)", 
            synthesize_cond4_full, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond4.wav"), hps.data.sampling_rate, audio_c4)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        import traceback
        traceback.print_exc()

print("All tasks finished.")

Loaded SID mapping for 16 files.
Prepared 16 tasks.
Sample task: uudb/tts1/data/test/wav/FJK_C051_118.wav|368|でも[なんかこう] そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた
Starting measurement loop...

Processing: FJK_C051_118 (SID: 368)
Text: でも[なんかこう] そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた
--- Cond 1 (Naive Mora Concat) ---
Audio duration: 11.38 seconds
Elapsed time: 1.5042 seconds
Real Time Factor (RTF): 0.1322
------------------------------
--- Cond 2 (Spec Naive Concat, chunk=1) ---
Audio duration: 7.12 seconds
Elapsed time: 0.8758 seconds
Real Time Factor (RTF): 0.1231
------------------------------
--- Cond 3 (Correlation OLA, chunk=1) ---
Audio duration: 7.12 seconds
Elapsed time: 0.9192 seconds
Real Time Factor (RTF): 0.1292
------------------------------
--- Cond 4 (Full Context Baseline) ---
Audio duration: 5.95 seconds
Elapsed time: 0.2140 seconds
Real Time Factor (RTF): 0.0360
------------------------------

Processing: FJK_C051_170 (SID: 368)
Text: やっぱ、やっぱディーが最後かな
--- Cond 1 (Naive Mo

# 文節単位で

In [57]:
import re
import pyopenjtalk

def get_bunsetsu_chunks_strict(text, phonemizer):
    """
    厳密な文節分割:
    - タグ [...] と通常テキストの境界で分割
    - 句読点 (、。) は直前の要素に結合して分割確定
    例: "でも[なんか]、そうすると" -> ["でも", "[なんか]、", "そうすると"]
    """
    # 1. トークン化: タグ、句読点、それ以外のテキストに分解
    # {cough} 等もタグ扱い
    tokens = re.split(r'({cough}|<cough>|\[.*?\]|[、。])', text)
    
    chunks_phonemes = []
    current_chunk_text = ""
    current_type = None # 'text' or 'tag'
    
    def finalize_chunk(txt):
        if not txt: return None
        
        # 音素化処理
        # 句読点が含まれているかチェックして sp に変換する処理が必要
        
        # 簡易パース: 末尾が句読点かどうか
        has_punct = False
        if txt.endswith("、") or txt.endswith("。"):
            has_punct = True
            content = txt[:-1] # 句読点除去
        else:
            content = txt
            
        if not content and not has_punct: return None
        
        # 中身の音素化
        ph_str = ""
        if content:
            # タグの場合
            if content.startswith("[") and content.endswith("]"):
                inner = content[1:-1]
                if inner:
                    k = pyopenjtalk.g2p(inner, kana=True).replace('ヲ', 'オ')
                    p = phonemizer(k)
                    ph_str = f"[ {p} ]"
                else:
                    ph_str = "[ ]"
            elif content in ["{cough}", "<cough>"]:
                ph_str = "<cough>"
            else:
                # 通常テキスト
                k = pyopenjtalk.g2p(content, kana=True).replace('ヲ', 'オ')
                p = phonemizer(k)
                ph_str = p
        
        # 句読点があれば sp を付与
        if has_punct:
            if ph_str:
                ph_str += " sp"
            else:
                ph_str = "sp"
                
        return ph_str.strip()

    for token in tokens:
        if not token or token.isspace():
            continue
            
        # トークンの種類判定
        is_punct = token in ["、", "。"]
        is_tag = (token.startswith("[") and token.endswith("]")) or token in ["{cough}", "<cough>"]
        
        if is_punct:
            # 句読点 -> 現在のチャンクに結合して、即座に確定(Flush)
            current_chunk_text += token
            
            ph = finalize_chunk(current_chunk_text)
            if ph: chunks_phonemes.append(ph)
            
            current_chunk_text = ""
            current_type = None
            
        else:
            # タグまたはテキスト
            new_type = 'tag' if is_tag else 'text'
            
            # タイプが変わったら、前のチャンクを確定して切り離す
            # (例: "でも" (text) -> "[なんか]" (tag) の境界)
            if current_type is not None and current_type != new_type:
                ph = finalize_chunk(current_chunk_text)
                if ph: chunks_phonemes.append(ph)
                current_chunk_text = ""
            
            current_chunk_text += token
            current_type = new_type
            
    # 残りがあれば処理
    if current_chunk_text:
        ph = finalize_chunk(current_chunk_text)
        if ph: chunks_phonemes.append(ph)
        
    return chunks_phonemes

In [58]:
def synthesize_cond1_bunsetsu(net_g, raw_text, sid, hps, phonemizer):
    """
    Cond 1: 文節単位単純接続 (Strict分割)
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # ★変更: Strict分割を使用
    bunsetsu_chunks = get_bunsetsu_chunks_strict(raw_text, phonemizer)
    
    audio_segments = []
    
    # 以下同様...
    for ph in bunsetsu_chunks:
        if not ph: continue
        stn_tst = get_text_from_phonemes(ph, hps)
        
        with torch.no_grad():
            x_tst = stn_tst.to(device).unsqueeze(0)
            x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
            
            audio = net_g.infer(
                x_tst, x_tst_lengths, sid=sid, 
                noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale
            )[0][0,0].data.cpu().float().numpy()
            
        audio_segments.append(audio)
    
    if not audio_segments: return np.array([])
    return np.concatenate(audio_segments)

In [59]:
def synthesize_cond2_bunsetsu(net_g, raw_text, sid, hps, phonemizer):
    """
    Cond 2: スペクトログラム単純接続 (Strict分割)
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # ★変更: Strict分割
    bunsetsu_chunks = get_bunsetsu_chunks_strict(raw_text, phonemizer)
    
    # 以下同様にリスト構築...
    all_phoneme_ids = []
    chunk_counts = []
    for ph in bunsetsu_chunks:
        if not ph: continue
        ids = get_text_from_phonemes(ph, hps)
        chunk_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids: return np.array([])
    
    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, 
            noise_scale, noise_scale_w, length_scale
        )
        w_ceil_flat = w_ceil.squeeze() 

        full_complex_spec = None
        current_ph_idx = 0
        current_z_frame = 0
        
        for count in chunk_counts:
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            
            z_end_frame = current_z_frame + z_len
            if z_end_frame > z.shape[2]: z_end_frame = z.shape[2]
            
            z_chunk = z[:, :, current_z_frame : z_end_frame]
            
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
            
            current_ph_idx += count
            current_z_frame = z_end_frame
            if current_z_frame >= z.shape[2]: break

        if full_complex_spec is None: return np.array([])
        return istft_finalize(net_g, full_complex_spec)

In [60]:
def synthesize_cond3_bunsetsu(net_g, raw_text, sid, hps, phonemizer, z_overlap_frames=5, search_range=2):
    """
    Cond 3: 相互相関OLA (全体エンコード -> 文節単位デコード + Overlap -> OLA)
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # 1. 文節リスト & ID化
    bunsetsu_chunks = get_bunsetsu_chunks_strict(raw_text, phonemizer)
    
    all_phoneme_ids = []
    chunk_counts = []
    for ph in bunsetsu_chunks:
        if not ph: continue
        ids = get_text_from_phonemes(ph, hps)
        chunk_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids: return np.array([])

    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        # 2. 全体エンコード
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, 
            noise_scale, noise_scale_w, length_scale
        )
        w_ceil_flat = w_ceil.squeeze()

        # 3. 文節ごとに OLA 処理
        full_complex_spec = None
        prev_raw_tail = None
        current_ph_idx = 0
        current_z_frame = 0
        ratio = None 
        
        for count in chunk_counts:
            # 長さ計算
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            
            # 切り出し範囲 (Overlap込み)
            z_end_nominal = current_z_frame + z_len
            z_end_decode = z_end_nominal + z_overlap_frames
            if z_end_decode > z.shape[2]: z_end_decode = z.shape[2]
            
            z_chunk = z[:, :, current_z_frame : z_end_decode]
            
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                if ratio is None:
                    ratio = complex_chunk.shape[-1] / z_chunk.shape[-1] if z_chunk.shape[-1] > 0 else 1.0
                
                # OLA (前回と同様のロジック)
                actual_z_overlap = max(0, z_chunk.shape[-1] - z_len)
                spec_overlap_len = int(actual_z_overlap * ratio)
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    if prev_raw_tail is None:
                        prev_ref = full_complex_spec[..., -spec_overlap_len:]
                    else:
                        prev_ref = prev_raw_tail
                        
                    curr_ref = complex_chunk[..., :spec_overlap_len]
                    valid_overlap = min(prev_ref.shape[-1], curr_ref.shape[-1])
                    
                    if valid_overlap > 0:
                        shift = 0
                        if valid_overlap > search_range * 2:
                            shift = find_best_frame_shift(
                                torch.abs(prev_ref[..., :valid_overlap]), 
                                torch.abs(curr_ref[..., :valid_overlap]), 
                                search_range
                            )
                        
                        start_off = max(0, min(-shift, search_range * 2))
                        aligned = complex_chunk[..., start_off:]
                        cross_len = min(valid_overlap, aligned.shape[-1])
                        
                        if cross_len > 0:
                            alpha = torch.linspace(0.0, 1.0, cross_len).to(device).view(1, 1, 1, cross_len)
                            merged = prev_ref[..., :cross_len] * (1 - alpha) + aligned[..., :cross_len] * alpha
                            full_complex_spec = torch.cat([
                                full_complex_spec[..., :-cross_len], merged, aligned[..., cross_len:]
                            ], dim=-1)
                        else:
                            full_complex_spec = torch.cat([full_complex_spec, aligned], dim=-1)
                    else:
                        full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
                
                # 末尾保存
                next_expected_overlap = int(z_overlap_frames * ratio)
                if complex_chunk.shape[-1] >= next_expected_overlap:
                    prev_raw_tail = complex_chunk[..., -next_expected_overlap:]
                else:
                    prev_raw_tail = complex_chunk

            current_ph_idx += count
            current_z_frame = z_end_nominal # 正味分だけ進める
            if current_z_frame >= z.shape[2]: break

        if full_complex_spec is None: return np.array([])
        return istft_finalize(net_g, full_complex_spec)

In [68]:

# ==========================================
# 実行準備
# ==========================================
# Phonemizerのインスタンス化 (ループの外で1回だけ行う)
phonemizer = Phonemizer()

# (以下、メインループ内での呼び出し例)
# ...
raw_japanese_text = "最近、インターステラーを見たのですけど、すごく面白かったです。" #(original_transcriptions_summary.txt等から取得)
#x_tst = stn_tst.to(device).unsqueeze(0)
#x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
sid = torch.LongTensor([374]).to(device)
# ...
audio_cond1 = synthesize_cond1_bunsetsu(net_g, raw_japanese_text, sid, hps, phonemizer)
audio_cond2 = synthesize_cond2_bunsetsu(net_g, raw_japanese_text, sid, hps, phonemizer)
audio_cond3 = synthesize_cond3_bunsetsu(net_g, raw_japanese_text, sid, hps, phonemizer,1,2)
# write(..., audio_cond1)
print("条件１")
display(Audio(audio_cond1, rate=hps.data.sampling_rate, normalize=False))
print("条件２")
display(Audio(audio_cond2, rate=hps.data.sampling_rate, normalize=False))
print("条件３")
display(Audio(audio_cond3, rate=hps.data.sampling_rate, normalize=False))

条件１


条件２


条件３


In [69]:
import os

# ==========================================
# 1. 日本語テキスト (書き起こし) の定義
# ==========================================
raw_input_text = """
uudb/tts1/data/test/wav/FJK_C051_118.wav
  Original: でも[なんかこう] そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた

uudb/tts1/data/test/wav/FJK_C051_170.wav
  Original: やっぱ、やっぱディーが最後かな

uudb/tts1/data/test/wav/FKC_C031_002.wav
  Original: [うんとね] 多分あたし一番持ってる気がするって感じ

uudb/tts1/data/test/wav/FMS_C051_072.wav
  Original: そうだよね、うんうん

uudb/tts1/data/test/wav/FMT_C041_134.wav
  Original: 何だそりゃ

uudb/tts1/data/test/wav/FMT_C041_259.wav
  Original: ディーはエーより後ってことだよね

uudb/tts1/data/test/wav/FTH_C004_044.wav
  Original: [えっとね] じゃあシー、行くね

uudb/tts1/data/test/wav/FTH_C005_152.wav
  Original: [あ] じゃあ、最初は、エーの[その]、交番の外にサラリーマンがいて

uudb/tts1/data/test/wav/FTS_C002_107.wav
  Original: [あ] ちゃんと入ってないんだ、[あー]

uudb/tts1/data/test/wav/FTS_C002_175.wav
  Original: [あー]、オッケーオッケー

uudb/tts1/data/test/wav/FTS_C004_126.wav
  Original: そうね、えっみたいな感じだよね

uudb/tts1/data/test/wav/FTS_C006_050.wav
  Original: なるほど、わかりました

uudb/tts1/data/test/wav/FTS_C007_137.wav
  Original: [あー]、見てるんだ

uudb/tts1/data/test/wav/FUE_C033_134.wav
  Original: [え]、わかんなくなってきた

uudb/tts1/data/test/wav/FYH_C042_090.wav
  Original: この、お母さんがすごい、あんまり上品でない食べ方をしてるのね

uudb/tts1/data/test/wav/FYH_C043_064.wav
  Original: はい、いいですか
"""

# ==========================================
# 2. 話者ID (SID) の参照データ定義
# ==========================================
sid_reference_data = """
uudb/tts1/data/test/wav/FJK_C051_118.wav|368|d e m o [ n a N k a k o: ] s o: s u r u t o sp m a t a m a t a a Q t a k a k u n a Q t a Q t e k o n o n a g a s i m a k a N t o k u g a y u Q t e r u N d a y o n e m a t a
uudb/tts1/data/test/wav/FJK_C051_170.wav|368|y a Q p a sp y a Q p a d i: g a s a i g o k a n a
uudb/tts1/data/test/wav/FKC_C031_002.wav|369|[ u N t o n e ] t a b u N a t a s i i t i b a N m o Q t e r u k i g a s u r u Q t e k a N z i
uudb/tts1/data/test/wav/FMS_C051_072.wav|370|s o: d a y o n e sp u N u N
uudb/tts1/data/test/wav/FMT_C041_134.wav|371|n a N d a s o ry a
uudb/tts1/data/test/wav/FMT_C041_259.wav|371|d i: w a e: y o r i a t o Q t e k o t o d a y o n e
uudb/tts1/data/test/wav/FTH_C004_044.wav|375|[ e Q t o n e ] zy a: s i: sp i k u n e
uudb/tts1/data/test/wav/FTH_C005_152.wav|375|[ a ] zy a: sp s a i sy o w a sp e: n o [ s o n o ] sp k o: b a N n o s o t o n i s a r a r i: m a N g a i t e
uudb/tts1/data/test/wav/FTS_C002_107.wav|376|[ a ] ch a N t o h a i Q t e n a i N d a sp [ a: ]
uudb/tts1/data/test/wav/FTS_C002_175.wav|376|[ a: ] sp o Q k e: o Q k e:
uudb/tts1/data/test/wav/FTS_C004_126.wav|376|s o: n e sp e Q m i t a i n a k a N z i d a y o n e
uudb/tts1/data/test/wav/FTS_C006_050.wav|376|n a r u h o d o sp w a k a r i m a s i t a
uudb/tts1/data/test/wav/FTS_C007_137.wav|376|[ a: ] sp m i t e r u N d a
uudb/tts1/data/test/wav/FUE_C033_134.wav|378|[ e ] sp w a k a N n a k u n a Q t e k i t a
uudb/tts1/data/test/wav/FYH_C042_090.wav|379|k o n o sp o k a: s a N g a s u g o i sp a N m a r i zy o: h i N d e n a i t a b e k a t a o s i t e r u n o n e
uudb/tts1/data/test/wav/FYH_C043_064.wav|379|h a i sp i: d e s u k a
"""

# ==========================================
# 3. データの紐付けとリスト生成処理
# ==========================================

def prepare_execution_list(raw_text, sid_data):
    # 1. SIDマップの作成 (ファイル名 -> SID)
    filename_to_sid = {}
    for line in sid_data.strip().split('\n'):
        line = line.strip()
        if not line: continue
        parts = line.split('|')
        if len(parts) >= 2:
            path = parts[0]
            sid = parts[1]
            fname = os.path.basename(path)
            filename_to_sid[fname] = int(sid)

    print(f"Loaded SID mapping for {len(filename_to_sid)} files.")

    # 2. テキストデータのパースと結合
    execution_lines = []
    current_wav = None
    
    for line in raw_text.strip().split('\n'):
        line = line.strip()
        if not line: continue
        
        if line.endswith('.wav'):
            current_wav = line
        elif line.startswith('Original:'):
            if current_wav:
                text = line.replace('Original:', '').strip()
                fname = os.path.basename(current_wav)
                
                # SIDの引き当て
                if fname in filename_to_sid:
                    sid_val = filename_to_sid[fname]
                    # フォーマット: パス|SID|テキスト
                    formatted_line = f"{current_wav}|{sid_val}|{text}"
                    execution_lines.append(formatted_line)
                else:
                    print(f"Warning: No SID found for {fname}, skipping.")
                
                current_wav = None

    return execution_lines

# リストの生成
lines = prepare_execution_list(raw_input_text, sid_reference_data)

print(f"Prepared {len(lines)} tasks.")
# 確認用出力 (最初の1件)
if lines:
    print("Sample task:", lines[0])

# ==========================================
# 4. メイン実行ループ
# ==========================================
# (以前のコードのループ部分をそのまま使用します)

print("Starting measurement loop...")

# チャンクサイズ設定 (Cond 2, 3用)
mora_chunk = 1

for i, line in enumerate(lines):
    line = line.strip()
    if not line: continue
    parts = line.split("|")
    
    # 読み込み
    file_path = parts[0]
    spk_id_val = int(parts[1])
    raw_text_jp = parts[2]
    
    filename = os.path.basename(file_path).replace(".wav", "")
    sid = torch.LongTensor([spk_id_val]).to(device)
    
    print(f"\nProcessing: {filename} (SID: {spk_id_val})")
    print(f"Text: {raw_text_jp}")

    try:
        # Cond 1: モーラ単位単純接続
        audio_c1 = measure_and_print(
            "Cond 1 (Naive Mora Concat)", 
            synthesize_cond1_bunsetsu, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)

        # Cond 2: スペクトログラム単純接続
        audio_c2 = measure_and_print(
            f"Cond 2 (Spec Naive Concat, chunk={mora_chunk})", 
            synthesize_cond2_bunsetsu, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond2.wav"), hps.data.sampling_rate, audio_c2)

        # Cond 3: 相互相関OLA
        audio_c3 = measure_and_print(
            f"Cond 3 (Correlation OLA, chunk={mora_chunk})", 
            synthesize_cond3_bunsetsu, 
            net_g, raw_text_jp, sid, hps, phonemizer, 
            1, 2   # overlap=5, search=2
        )
        write(os.path.join(output_dir, f"{filename}_cond3.wav"), hps.data.sampling_rate, audio_c3)

        # Cond 4: 全体一括 (Baseline)
        audio_c4 = measure_and_print(
            "Cond 4 (Full Context Baseline)", 
            synthesize_cond4_full, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond4.wav"), hps.data.sampling_rate, audio_c4)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        import traceback
        traceback.print_exc()

print("All tasks finished.")

Loaded SID mapping for 16 files.
Prepared 16 tasks.
Sample task: uudb/tts1/data/test/wav/FJK_C051_118.wav|368|でも[なんかこう] そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた
Starting measurement loop...

Processing: FJK_C051_118 (SID: 368)
Text: でも[なんかこう] そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた
--- Cond 1 (Naive Mora Concat) ---
Audio duration: 6.11 seconds
Elapsed time: 0.2690 seconds
Real Time Factor (RTF): 0.0440
------------------------------
--- Cond 2 (Spec Naive Concat, chunk=1) ---
Audio duration: 5.94 seconds
Elapsed time: 0.2615 seconds
Real Time Factor (RTF): 0.0440
------------------------------
--- Cond 3 (Correlation OLA, chunk=1) ---
Audio duration: 5.96 seconds
Elapsed time: 0.2221 seconds
Real Time Factor (RTF): 0.0373
------------------------------
--- Cond 4 (Full Context Baseline) ---
Audio duration: 5.47 seconds
Elapsed time: 0.2668 seconds
Real Time Factor (RTF): 0.0488
------------------------------

Processing: FJK_C051_170 (SID: 368)
Text: やっぱ、やっぱディーが最後かな
--- Cond 1 (Naive Mor

# mecab利用

In [13]:
import re
import MeCab
import pyopenjtalk

# ==========================================
# 1. MeCabによる文節分割ロジック
# ==========================================
def split_text_to_bunsetsu(text):
    """
    MeCabを用いてテキストを「文節」単位のリストに分割する。
    文節 = [自立語 + 付属語...] の塊
    """
    # unidic-lite を使用する場合の標準的なタガー初期化
    tagger = MeCab.Tagger()
    
    node = tagger.parseToNode(text)
    chunks = []
    current_chunk = ""
    
    while node:
        if node.surface == "":  # BOS/EOSスキップ
            node = node.next
            continue
            
        # 品詞情報を取得
        # UniDic系: pos1, pos2, pos3, pos4, ...
        # IPADIC系: pos1, pos2, ...
        features = node.feature.split(",")
        pos1 = features[0]  # 品詞大分類
        pos2 = features[1]  # 品詞中分類
        
        # --- 文節の切れ目判定 ---
        # 「自立語」が来たタイミングで分割（ただし文頭は除く）
        # ※定義は簡易的なものですが、音声合成の韻律単位としては概ね機能します
        is_independent = False
        
        # 分割候補となる品詞 (名詞, 動詞, 形容詞, 副詞, 連体詞, 接続詞, 感動詞, 接頭詞, 形状詞)
        if pos1 in ["名詞", "動詞", "形容詞", "副詞", "連体詞", "接続詞", "感動詞", "接頭詞", "形状詞", "代名詞"]:
            # ただし「非自立」「接尾」などは前の文節に付ける（分割しない）
            if pos2 not in ["非自立", "接尾"]:
                is_independent = True
                
        # 代名詞などの扱い（UniDicでは代名詞も独立しやすいが、IPADICでは名詞扱い）
        
        # 分割実行
        if is_independent and current_chunk:
            chunks.append(current_chunk)
            current_chunk = ""
            
        current_chunk += node.surface
        node = node.next
        
    # 残りを追加
    if current_chunk:
        chunks.append(current_chunk)
        
    return chunks

# ==========================================
# 2. タグや句読点処理との統合
# ==========================================
def get_bunsetsu_chunks_mecab(text, phonemizer):
    """
    タグ [...] や句読点を考慮しつつ、
    通常テキスト部分は MeCab で文節分割して音素化リストを返す
    """
    # 1. タグや句読点で大まかに分割
    tokens = re.split(r'({cough}|<cough>|\[.*?\]|[、。])', text)
    
    final_phoneme_chunks = []
    
    # 一時バッファ（MeCabにかける前のテキスト）
    text_buffer = ""
    
    def flush_text_buffer():
        nonlocal text_buffer
        if not text_buffer: return
        
        # MeCabで文節分割
        bunsetsu_list = split_text_to_bunsetsu(text_buffer)
        
        for b_text in bunsetsu_list:
            # 音素化
            k = pyopenjtalk.g2p(b_text, kana=True).replace('ヲ', 'オ')
            p = phonemizer(k)
            if p.strip():
                final_phoneme_chunks.append(p.strip())
        
        text_buffer = ""

    for token in tokens:
        if not token or token.isspace():
            continue
            
        # A. 句読点 (直前の文節にくっつける -> sp化)
        if token in ["、", "。"]:
            # バッファがあれば先に吐き出す
            if text_buffer:
                # 最後の文節を取得して sp を付けるために、ここで自前処理
                bunsetsu_list = split_text_to_bunsetsu(text_buffer)
                for i, b_text in enumerate(bunsetsu_list):
                    k = pyopenjtalk.g2p(b_text, kana=True).replace('ヲ', 'オ')
                    p = phonemizer(k)
                    if p.strip():
                        # 最後の文節なら sp を付与
                        if i == len(bunsetsu_list) - 1:
                            final_phoneme_chunks.append(p.strip() + " sp")
                        else:
                            final_phoneme_chunks.append(p.strip())
                text_buffer = ""
            else:
                # 直前の既存チャンクに sp を追加
                if final_phoneme_chunks:
                    final_phoneme_chunks[-1] += " sp"
                else:
                    final_phoneme_chunks.append("sp")
            continue
            
        # B. タグ
        if (token.startswith("[") and token.endswith("]")) or token in ["{cough}", "<cough>"]:
            flush_text_buffer() # テキストがあれば先に処理
            
            # タグの処理
            if token.startswith("["):
                content = token[1:-1]
                if content:
                    k = pyopenjtalk.g2p(content, kana=True).replace('ヲ', 'オ')
                    p = phonemizer(k)
                    final_phoneme_chunks.append(f"[ {p} ]")
                else:
                    final_phoneme_chunks.append("[ ]")
            else:
                final_phoneme_chunks.append("<cough>")
            continue
            
        # C. 通常テキスト -> バッファに溜める
        text_buffer += token
        
    # 残りのテキストを処理
    flush_text_buffer()
    
    return final_phoneme_chunks

In [14]:
def synthesize_cond1_bunsetsu_M(net_g, raw_text, sid, hps, phonemizer):
    """
    Cond 1: 文節単位単純接続 (Strict分割)
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # ★変更: Strict分割を使用
    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    audio_segments = []
    
    # 以下同様...
    for ph in bunsetsu_chunks:
        if not ph: continue
        stn_tst = get_text_from_phonemes(ph, hps)
        
        with torch.no_grad():
            x_tst = stn_tst.to(device).unsqueeze(0)
            x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
            
            audio = net_g.infer(
                x_tst, x_tst_lengths, sid=sid, 
                noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale
            )[0][0,0].data.cpu().float().numpy()
            
        audio_segments.append(audio)
    
    if not audio_segments: return np.array([])
    return np.concatenate(audio_segments)

In [16]:
def synthesize_cond2_bunsetsu_M(net_g, raw_text, sid, hps, phonemizer):
    """
    Cond 2: スペクトログラム単純接続 (Strict分割)
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # ★変更: Strict分割
    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    # 以下同様にリスト構築...
    all_phoneme_ids = []
    chunk_counts = []
    for ph in bunsetsu_chunks:
        if not ph: continue
        ids = get_text_from_phonemes(ph, hps)
        chunk_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids: return np.array([])
    
    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, 
            noise_scale, noise_scale_w, length_scale
        )
        w_ceil_flat = w_ceil.squeeze() 

        full_complex_spec = None
        current_ph_idx = 0
        current_z_frame = 0
        
        for count in chunk_counts:
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            
            z_end_frame = current_z_frame + z_len
            if z_end_frame > z.shape[2]: z_end_frame = z.shape[2]
            
            z_chunk = z[:, :, current_z_frame : z_end_frame]
            
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
            
            current_ph_idx += count
            current_z_frame = z_end_frame
            if current_z_frame >= z.shape[2]: break

        if full_complex_spec is None: return np.array([])
        return istft_finalize(net_g, full_complex_spec)

In [17]:
def synthesize_cond3_bunsetsu_M(net_g, raw_text, sid, hps, phonemizer, z_overlap_frames=5, search_range=2):
    """
    Cond 3: 相互相関OLA (全体エンコード -> 文節単位デコード + Overlap -> OLA)
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # 1. 文節リスト & ID化
    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    all_phoneme_ids = []
    chunk_counts = []
    for ph in bunsetsu_chunks:
        if not ph: continue
        ids = get_text_from_phonemes(ph, hps)
        chunk_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids: return np.array([])

    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        # 2. 全体エンコード
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, 
            noise_scale, noise_scale_w, length_scale
        )
        w_ceil_flat = w_ceil.squeeze()

        # 3. 文節ごとに OLA 処理
        full_complex_spec = None
        prev_raw_tail = None
        current_ph_idx = 0
        current_z_frame = 0
        ratio = None 
        
        for count in chunk_counts:
            # 長さ計算
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            
            # 切り出し範囲 (Overlap込み)
            z_end_nominal = current_z_frame + z_len
            z_end_decode = z_end_nominal + z_overlap_frames
            if z_end_decode > z.shape[2]: z_end_decode = z.shape[2]
            
            z_chunk = z[:, :, current_z_frame : z_end_decode]
            
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                if ratio is None:
                    ratio = complex_chunk.shape[-1] / z_chunk.shape[-1] if z_chunk.shape[-1] > 0 else 1.0
                
                # OLA (前回と同様のロジック)
                actual_z_overlap = max(0, z_chunk.shape[-1] - z_len)
                spec_overlap_len = int(actual_z_overlap * ratio)
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    if prev_raw_tail is None:
                        prev_ref = full_complex_spec[..., -spec_overlap_len:]
                    else:
                        prev_ref = prev_raw_tail
                        
                    curr_ref = complex_chunk[..., :spec_overlap_len]
                    valid_overlap = min(prev_ref.shape[-1], curr_ref.shape[-1])
                    
                    if valid_overlap > 0:
                        shift = 0
                        if valid_overlap > search_range * 2:
                            shift = find_best_frame_shift(
                                torch.abs(prev_ref[..., :valid_overlap]), 
                                torch.abs(curr_ref[..., :valid_overlap]), 
                                search_range
                            )
                        
                        start_off = max(0, min(-shift, search_range * 2))
                        aligned = complex_chunk[..., start_off:]
                        cross_len = min(valid_overlap, aligned.shape[-1])
                        
                        if cross_len > 0:
                            alpha = torch.linspace(0.0, 1.0, cross_len).to(device).view(1, 1, 1, cross_len)
                            merged = prev_ref[..., :cross_len] * (1 - alpha) + aligned[..., :cross_len] * alpha
                            full_complex_spec = torch.cat([
                                full_complex_spec[..., :-cross_len], merged, aligned[..., cross_len:]
                            ], dim=-1)
                        else:
                            full_complex_spec = torch.cat([full_complex_spec, aligned], dim=-1)
                    else:
                        full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
                
                # 末尾保存
                next_expected_overlap = int(z_overlap_frames * ratio)
                if complex_chunk.shape[-1] >= next_expected_overlap:
                    prev_raw_tail = complex_chunk[..., -next_expected_overlap:]
                else:
                    prev_raw_tail = complex_chunk

            current_ph_idx += count
            current_z_frame = z_end_nominal # 正味分だけ進める
            if current_z_frame >= z.shape[2]: break

        if full_complex_spec is None: return np.array([])
        return istft_finalize(net_g, full_complex_spec)

In [22]:

# ==========================================
# 実行準備
# ==========================================
# Phonemizerのインスタンス化 (ループの外で1回だけ行う)
phonemizer = Phonemizer()

# (以下、メインループ内での呼び出し例)
# ...
raw_japanese_text = "[あ]ちゃんと入ってないんだ 、 [あー]" #(original_transcriptions_summary.txt等から取得)
#x_tst = stn_tst.to(device).unsqueeze(0)
#x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
sid = torch.LongTensor([374]).to(device)
# ...
audio_cond1 = synthesize_cond1_bunsetsu_M(net_g, raw_japanese_text, sid, hps, phonemizer)
audio_cond2 = synthesize_cond2_bunsetsu_M(net_g, raw_japanese_text, sid, hps, phonemizer)
audio_cond3 = synthesize_cond3_bunsetsu_M(net_g, raw_japanese_text, sid, hps, phonemizer,1,2)
# write(..., audio_cond1)
print("条件１")
display(Audio(audio_cond1, rate=hps.data.sampling_rate, normalize=False))
print("条件２")
display(Audio(audio_cond2, rate=hps.data.sampling_rate, normalize=False))
print("条件３")
display(Audio(audio_cond3, rate=hps.data.sampling_rate, normalize=False))

条件１


条件２


条件３


In [28]:
import os

# ==========================================
# 1. 日本語テキスト (書き起こし) の定義
# ==========================================
raw_input_text = """
uudb/tts1/data/test/wav/FJK_C051_118.wav
  Original: でも[なんかこう]そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた

uudb/tts1/data/test/wav/FJK_C051_170.wav
  Original: やっぱ、やっぱディーが最後かな

uudb/tts1/data/test/wav/FKC_C031_002.wav
  Original: [うんとね]多分あたし一番持ってる気がするって感じ

uudb/tts1/data/test/wav/FMS_C051_072.wav
  Original: そうだよね、うんうん

uudb/tts1/data/test/wav/FMT_C041_134.wav
  Original: 何だそりゃ

uudb/tts1/data/test/wav/FMT_C041_259.wav
  Original: ディーはエーより後ってことだよね

uudb/tts1/data/test/wav/FTH_C004_044.wav
  Original: [えっとね]じゃあシー、行くね

uudb/tts1/data/test/wav/FTH_C005_152.wav
  Original: [あ]じゃあ、最初は、エーの[その]、交番の外にサラリーマンがいて

uudb/tts1/data/test/wav/FTS_C002_107.wav
  Original: [あ]ちゃんと入ってないんだ、[あー]

uudb/tts1/data/test/wav/FTS_C002_175.wav
  Original: [あー]、オッケーオッケー

uudb/tts1/data/test/wav/FTS_C004_126.wav
  Original: そうね、えっみたいな感じだよね

uudb/tts1/data/test/wav/FTS_C006_050.wav
  Original: なるほど、わかりました

uudb/tts1/data/test/wav/FTS_C007_137.wav
  Original: [あー]、見てるんだ

uudb/tts1/data/test/wav/FUE_C033_134.wav
  Original: [え]、わかんなくなってきた

uudb/tts1/data/test/wav/FYH_C042_090.wav
  Original: この、お母さんがすごい、あんまり上品でない食べ方をしてるのね

uudb/tts1/data/test/wav/FYH_C043_064.wav
  Original: はい、いいですか
"""

# ==========================================
# 2. 話者ID (SID) の参照データ定義
# ==========================================
sid_reference_data = """
uudb/tts1/data/test/wav/FJK_C051_118.wav|368|d e m o [ n a N k a k o: ] s o: s u r u t o sp m a t a m a t a a Q t a k a k u n a Q t a Q t e k o n o n a g a s i m a k a N t o k u g a y u Q t e r u N d a y o n e m a t a
uudb/tts1/data/test/wav/FJK_C051_170.wav|368|y a Q p a sp y a Q p a d i: g a s a i g o k a n a
uudb/tts1/data/test/wav/FKC_C031_002.wav|369|[ u N t o n e ] t a b u N a t a s i i t i b a N m o Q t e r u k i g a s u r u Q t e k a N z i
uudb/tts1/data/test/wav/FMS_C051_072.wav|370|s o: d a y o n e sp u N u N
uudb/tts1/data/test/wav/FMT_C041_134.wav|371|n a N d a s o ry a
uudb/tts1/data/test/wav/FMT_C041_259.wav|371|d i: w a e: y o r i a t o Q t e k o t o d a y o n e
uudb/tts1/data/test/wav/FTH_C004_044.wav|375|[ e Q t o n e ] zy a: s i: sp i k u n e
uudb/tts1/data/test/wav/FTH_C005_152.wav|375|[ a ] zy a: sp s a i sy o w a sp e: n o [ s o n o ] sp k o: b a N n o s o t o n i s a r a r i: m a N g a i t e
uudb/tts1/data/test/wav/FTS_C002_107.wav|376|[ a ] ch a N t o h a i Q t e n a i N d a sp [ a: ]
uudb/tts1/data/test/wav/FTS_C002_175.wav|376|[ a: ] sp o Q k e: o Q k e:
uudb/tts1/data/test/wav/FTS_C004_126.wav|376|s o: n e sp e Q m i t a i n a k a N z i d a y o n e
uudb/tts1/data/test/wav/FTS_C006_050.wav|376|n a r u h o d o sp w a k a r i m a s i t a
uudb/tts1/data/test/wav/FTS_C007_137.wav|376|[ a: ] sp m i t e r u N d a
uudb/tts1/data/test/wav/FUE_C033_134.wav|378|[ e ] sp w a k a N n a k u n a Q t e k i t a
uudb/tts1/data/test/wav/FYH_C042_090.wav|379|k o n o sp o k a: s a N g a s u g o i sp a N m a r i zy o: h i N d e n a i t a b e k a t a o s i t e r u n o n e
uudb/tts1/data/test/wav/FYH_C043_064.wav|379|h a i sp i: d e s u k a
"""

# ==========================================
# 3. データの紐付けとリスト生成処理
# ==========================================

def prepare_execution_list(raw_text, sid_data):
    # 1. SIDマップの作成 (ファイル名 -> SID)
    filename_to_sid = {}
    for line in sid_data.strip().split('\n'):
        line = line.strip()
        if not line: continue
        parts = line.split('|')
        if len(parts) >= 2:
            path = parts[0]
            sid = parts[1]
            fname = os.path.basename(path)
            filename_to_sid[fname] = int(sid)

    print(f"Loaded SID mapping for {len(filename_to_sid)} files.")

    # 2. テキストデータのパースと結合
    execution_lines = []
    current_wav = None
    
    for line in raw_text.strip().split('\n'):
        line = line.strip()
        if not line: continue
        
        if line.endswith('.wav'):
            current_wav = line
        elif line.startswith('Original:'):
            if current_wav:
                text = line.replace('Original:', '').strip()
                fname = os.path.basename(current_wav)
                
                # SIDの引き当て
                if fname in filename_to_sid:
                    sid_val = filename_to_sid[fname]
                    # フォーマット: パス|SID|テキスト
                    formatted_line = f"{current_wav}|{sid_val}|{text}"
                    execution_lines.append(formatted_line)
                else:
                    print(f"Warning: No SID found for {fname}, skipping.")
                
                current_wav = None

    return execution_lines

# リストの生成
lines = prepare_execution_list(raw_input_text, sid_reference_data)

print(f"Prepared {len(lines)} tasks.")
# 確認用出力 (最初の1件)
if lines:
    print("Sample task:", lines[0])

# ==========================================
# 4. メイン実行ループ
# ==========================================
# (以前のコードのループ部分をそのまま使用します)

print("Starting measurement loop...")

# チャンクサイズ設定 (Cond 2, 3用)
mora_chunk = 1

for i, line in enumerate(lines):
    line = line.strip()
    if not line: continue
    parts = line.split("|")
    
    # 読み込み
    file_path = parts[0]
    spk_id_val = int(parts[1])
    raw_text_jp = parts[2]
    
    filename = os.path.basename(file_path).replace(".wav", "")
    sid = torch.LongTensor([spk_id_val]).to(device)
    
    print(f"\nProcessing: {filename} (SID: {spk_id_val})")
    print(f"Text: {raw_text_jp}")

    try:
        # Cond 1: モーラ単位単純接続
        audio_c1 = measure_and_print(
            "Cond 1 (Naive Mora Concat)", 
            synthesize_cond1_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)

        # Cond 2: スペクトログラム単純接続
        audio_c2 = measure_and_print(
            f"Cond 2 (Spec Naive Concat, chunk={mora_chunk})", 
            synthesize_cond2_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond2.wav"), hps.data.sampling_rate, audio_c2)

        # Cond 3: 相互相関OLA
        audio_c3 = measure_and_print(
            f"Cond 3 (Correlation OLA, chunk={mora_chunk})", 
            synthesize_cond3_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer, 
            1, 2   # overlap=5, search=2
        )
        write(os.path.join(output_dir, f"{filename}_cond3.wav"), hps.data.sampling_rate, audio_c3)

        # Cond 4: 全体一括 (Baseline)
        audio_c4 = measure_and_print(
            "Cond 4 (Full Context Baseline)", 
            synthesize_cond4_full, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond4.wav"), hps.data.sampling_rate, audio_c4)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        import traceback
        traceback.print_exc()

print("All tasks finished.")

Loaded SID mapping for 16 files.
Prepared 16 tasks.
Sample task: uudb/tts1/data/test/wav/FJK_C051_118.wav|368|でも[なんかこう]そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた
Starting measurement loop...

Processing: FJK_C051_118 (SID: 368)
Text: でも[なんかこう]そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた
--- Cond 1 (Naive Mora Concat) ---
Audio duration: 7.58 seconds
Elapsed time: 0.5622 seconds
Real Time Factor (RTF): 0.0741
------------------------------
--- Cond 2 (Spec Naive Concat, chunk=1) ---
Audio duration: 6.27 seconds
Elapsed time: 0.4544 seconds
Real Time Factor (RTF): 0.0725
------------------------------
--- Cond 3 (Correlation OLA, chunk=1) ---
Audio duration: 6.28 seconds
Elapsed time: 0.3834 seconds
Real Time Factor (RTF): 0.0611
------------------------------
--- Cond 4 (Full Context Baseline) ---
Audio duration: 5.95 seconds
Elapsed time: 0.1929 seconds
Real Time Factor (RTF): 0.0324
------------------------------

Processing: FJK_C051_170 (SID: 368)
Text: やっぱ、やっぱディーが最後かな
--- Cond 1 (Naive Mora 

In [29]:
import torch
import numpy as np

# ==========================================
# 1. 共通準備: 潜在表現と分割情報の生成
# ==========================================
def prepare_shared_latents(net_g, raw_text, sid, hps, phonemizer):
    """
    Cond 2, 3, 4 で共有する潜在表現 z と、文節分割情報を一括生成する
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    # A. 文節分割 (MeCab版を使用)
    # ※ get_bunsetsu_chunks_mecab が定義済みであることを前提とします
    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    # B. 全体の音素列IDと、各文節の音素数を取得
    all_phoneme_ids = []
    chunk_phoneme_counts = []
    
    for ph in bunsetsu_chunks:
        if not ph: continue
        ids = get_text_from_phonemes(ph, hps)
        chunk_phoneme_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids:
        return None, None, None, None

    # C. 一括エンコード (Full Context)
    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        # エンコーダ実行 (z, w_ceil, g を取得)
        # ※ get_z_and_phoneme_durations は修正済みのものを使用
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, 
            noise_scale, noise_scale_w, length_scale
        )
    
    return z, w_ceil, g, chunk_phoneme_counts

# ==========================================
# 2. Cond 2: スペクトログラム単純接続 (共有z版)
# ==========================================
def synthesize_cond2_shared(net_g, z, w_ceil, g, chunk_counts):
    """
    共有された z を文節ごとに切り出し、デコードして単純結合する
    """
    if z is None: return np.array([])
    
    w_ceil_flat = w_ceil.squeeze() 
    full_complex_spec = None
    
    current_ph_idx = 0
    current_z_frame = 0
    
    with torch.no_grad():
        for count in chunk_counts:
            # この文節に対応するフレーム数を計算
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            
            # z 切り出し
            z_end_frame = current_z_frame + z_len
            if z_end_frame > z.shape[2]: z_end_frame = z.shape[2]
            
            z_chunk = z[:, :, current_z_frame : z_end_frame]
            
            # デコード
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                # 単純結合
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
            
            # ポインタ更新
            current_ph_idx += count
            current_z_frame = z_end_frame
            if current_z_frame >= z.shape[2]: break

    if full_complex_spec is None: return np.array([])
    return istft_finalize(net_g, full_complex_spec)

# ==========================================
# 3. Cond 3: 相互相関OLA (共有z版)
# ==========================================
def synthesize_cond3_shared(net_g, z, w_ceil, g, chunk_counts, z_overlap_frames=5, search_range=2):
    """
    共有された z をOverlap込みで切り出し、サブバンド相互相関で位置合わせしてOLA
    """
    if z is None: return np.array([])

    device = z.device
    w_ceil_flat = w_ceil.squeeze()
    
    full_complex_spec = None
    prev_raw_tail = None      # OLA用の前回の末尾(重複部)
    prev_chunk_complex = None # シフト探索用の前回の複素スペクトル全体(または末尾)
    
    current_ph_idx = 0
    current_z_frame = 0
    ratio = None
    
    with torch.no_grad():
        for count in chunk_counts:
            # 1. この文節の正味の長さ (zフレーム)
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            
            # 2. 切り出し範囲 (Overlap込み)
            z_end_nominal = current_z_frame + z_len
            z_end_decode = z_end_nominal + z_overlap_frames
            if z_end_decode > z.shape[2]: z_end_decode = z.shape[2]
            
            z_chunk = z[:, :, current_z_frame : z_end_decode]
            
            # 3. デコード
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                if ratio is None:
                    # アップサンプリング率の推定
                    ratio = complex_chunk.shape[-1] / z_chunk.shape[-1] if z_chunk.shape[-1] > 0 else 1.0
                
                # --- シフト探索 (サブバンド相関) ---
                shift = 0
                if prev_chunk_complex is not None:
                    # 前回の末尾と今回の先頭を使ってシフト量を計算
                    # ※ find_best_frame_shift_subband は直前に定義した関数
                    shift = find_best_frame_shift_subband(
                        prev_chunk_complex, 
                        complex_chunk, 
                        search_range
                    )
                
                # --- OLA処理 ---
                actual_z_overlap = max(0, z_chunk.shape[-1] - z_len)
                spec_overlap_len = int(actual_z_overlap * ratio)
                
                # シフト適用 (簡単のため、今回は複素スペクトル全体をシフトして切り出し位置を調整)
                # OLAに必要な部分を抽出
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    # 前回の末尾 (prev_raw_tail) と 今回の先頭 (curr_ref) をクロスフェード
                    
                    # shift > 0 (未来へズレる) -> 今回の開始を遅らせる (先頭を削る)
                    # shift < 0 (過去へズレる) -> 今回の開始を早める (前回の末尾に被せる深さを増やす...が、
                    # ここでは単純に「今回生成された波形」を shift 分だけずらして結合します)
                    
                    # 簡易実装: shift分だけ complex_chunk の読み出し開始位置をずらす
                    # shift正: 今回の波形が遅れているべき -> 先頭を余分に捨てる or 前回の末尾を伸ばす?
                    # ここでは「位相が合う点」を探したので、complex_chunk を shift シフトさせれば合うはず。
                    
                    # shift の符号定義: find_best... で「前に対して次がどれだけズレているか」
                    # positive shift: 次が遅れている -> 次の波形を左(過去)に寄せる必要がある?
                    # いや、相互相関のピーク位置 s は、f(t) と g(t+s) がマッチすることを意味します。
                    # なので、g (今回の波形) を -s だけずらす(スライスする)のが正解。
                    
                    # 安全マージンをとってスライス
                    start_idx = 0
                    if shift != 0:
                        # shift が正なら、今回の波形の後ろの方がマッチする -> 先頭を削る
                        if shift > 0:
                            start_idx = shift
                        # shift が負なら、今回の波形の手前がマッチする -> 本来はパディングが必要だが、
                        # 探索範囲が狭いので 0 スタートにする(無視)か、前回の末尾を削る等の調整が必要。
                        # ここでは簡易的に start_idx = 0 (負の場合は補正なし) とします
                        else:
                            start_idx = 0
                    
                    curr_ref = complex_chunk[..., start_idx : start_idx + spec_overlap_len]
                    
                    # クロスフェード長
                    cross_len = min(prev_raw_tail.shape[-1], curr_ref.shape[-1])
                    
                    if cross_len > 0:
                        alpha = torch.linspace(0.0, 1.0, cross_len).to(device).view(1, 1, 1, cross_len)
                        
                        # 前回の末尾 vs 今回の先頭
                        merged = prev_raw_tail[..., :cross_len] * (1 - alpha) + curr_ref[..., :cross_len] * alpha
                        
                        # 結合: [全体] + [マージ部] + [今回の残り]
                        full_complex_spec = torch.cat([
                            full_complex_spec[..., :-cross_len], # 前回のマージ部手前まで
                            merged,
                            complex_chunk[..., start_idx + cross_len:]
                        ], dim=-1)
                    else:
                        # 重なりが確保できない場合は単純結合
                        full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)

                # 次回用に保存
                # 次のオーバーラップ期待値
                next_expected_overlap = int(z_overlap_frames * ratio)
                
                # シフト計算用に「今回の生成結果そのもの」を保存
                prev_chunk_complex = complex_chunk
                
                # OLA用に「今回の末尾」を保存
                if complex_chunk.shape[-1] >= next_expected_overlap:
                    prev_raw_tail = complex_chunk[..., -next_expected_overlap:]
                else:
                    prev_raw_tail = complex_chunk

            # ポインタ更新
            current_ph_idx += count
            current_z_frame = z_end_nominal # 正味分だけ進める
            if current_z_frame >= z.shape[2]: break

    if full_complex_spec is None: return np.array([])
    return istft_finalize(net_g, full_complex_spec)

# ==========================================
# 4. Cond 4: 全体一括生成 (共有z版)
# ==========================================
def synthesize_cond4_shared(net_g, z, g):
    """
    共有された z をそのまま一括デコード (Topline)
    """
    if z is None: return np.array([])
    
    with torch.no_grad():
        _, _, spec, phase = net_g.dec(z, g=g)
        full_complex_spec = spec * torch.exp(1j * phase)
        
    return istft_finalize(net_g, full_complex_spec)

# 更新

In [30]:
# ==========================================
# Cond 1: 文節単位単純接続 (レイテンシ計測版)
# ==========================================
def synthesize_cond1_bunsetsu_M(net_g, raw_text, sid, hps, phonemizer):
    device = next(net_g.parameters()).device
    sid = sid.to(device)
    
    # 文節分割
    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    audio_segments = []
    first_latency = None
    process_start = time.time() # 計測開始
    
    for i, ph in enumerate(bunsetsu_chunks):
        if not ph: continue
        
        stn_tst = get_text_from_phonemes(ph, hps)
        
        with torch.no_grad():
            x_tst = stn_tst.to(device).unsqueeze(0)
            x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
            
            # 生成
            audio = net_g.infer(
                x_tst, x_tst_lengths, sid=sid, 
                noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale
            )[0][0,0].data.cpu().float().numpy()
            
            # ★最初のチャンク生成が終わった時点で時間を記録
            if first_latency is None:
                if torch.cuda.is_available(): torch.cuda.synchronize()
                first_latency = time.time() - process_start

        audio_segments.append(audio)
    
    if not audio_segments: return np.array([]), 0.0
    
    # 結合して返す + レイテンシ
    return np.concatenate(audio_segments), (first_latency if first_latency else 0.0)


# ==========================================
# Cond 2: スペクトログラム単純接続 (レイテンシ計測版)
# ==========================================
def synthesize_cond2_bunsetsu_M(net_g, raw_text, sid, hps, phonemizer):
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    # 全体ID化
    all_phoneme_ids = []
    chunk_counts = []
    for ph in bunsetsu_chunks:
        if not ph: continue
        ids = get_text_from_phonemes(ph, hps)
        chunk_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids: return np.array([]), 0.0

    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        process_start = time.time() # 計測開始
        
        # 1. 全体エンコード
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, 
            noise_scale, noise_scale_w, length_scale
        )
        w_ceil_flat = w_ceil.squeeze() 

        full_complex_spec = None
        current_ph_idx = 0
        current_z_frame = 0
        first_latency = None
        
        # 2. 文節ごとにデコード
        for count in chunk_counts:
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            
            z_end_frame = current_z_frame + z_len
            if z_end_frame > z.shape[2]: z_end_frame = z.shape[2]
            z_chunk = z[:, :, current_z_frame : z_end_frame]
            
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                # ★最初のスペクトログラム生成が終わった時点を「First Audio Ready」とみなす
                # (厳密にはiSTFT時間が加わりますが、NN推論に比べれば微小なため近似とします)
                if first_latency is None:
                    if torch.cuda.is_available(): torch.cuda.synchronize()
                    first_latency = time.time() - process_start
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
            
            current_ph_idx += count
            current_z_frame = z_end_frame
            if current_z_frame >= z.shape[2]: break

        if full_complex_spec is None: return np.array([]), 0.0
        
        audio = istft_finalize(net_g, full_complex_spec)
        return audio, (first_latency if first_latency else 0.0)


# ==========================================
# Cond 3: 相互相関OLA (レイテンシ計測版)
# ==========================================
def synthesize_cond3_bunsetsu_M(net_g, raw_text, sid, hps, phonemizer, z_overlap_frames=5, search_range=2):
    device = next(net_g.parameters()).device
    sid = sid.to(device)

    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    all_phoneme_ids = []
    chunk_counts = []
    for ph in bunsetsu_chunks:
        if not ph: continue
        ids = get_text_from_phonemes(ph, hps)
        chunk_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids: return np.array([]), 0.0

    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        process_start = time.time() # 計測開始
        
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, 
            noise_scale, noise_scale_w, length_scale
        )
        w_ceil_flat = w_ceil.squeeze()

        full_complex_spec = None
        prev_raw_tail = None
        current_ph_idx = 0
        current_z_frame = 0
        first_latency = None
        ratio = None 
        
        for count in chunk_counts:
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            
            z_end_nominal = current_z_frame + z_len
            z_end_decode = z_end_nominal + z_overlap_frames
            if z_end_decode > z.shape[2]: z_end_decode = z.shape[2]
            z_chunk = z[:, :, current_z_frame : z_end_decode]
            
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                # ★最初のチャンクのデコード完了時点をレイテンシとする
                if first_latency is None:
                    if torch.cuda.is_available(): torch.cuda.synchronize()
                    first_latency = time.time() - process_start

                if ratio is None:
                    ratio = complex_chunk.shape[-1] / z_chunk.shape[-1] if z_chunk.shape[-1] > 0 else 1.0
                
                # OLA処理
                actual_z_overlap = max(0, z_chunk.shape[-1] - z_len)
                spec_overlap_len = int(actual_z_overlap * ratio)
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    if prev_raw_tail is None:
                        prev_ref = full_complex_spec[..., -spec_overlap_len:]
                    else:
                        prev_ref = prev_raw_tail
                        
                    curr_ref = complex_chunk[..., :spec_overlap_len]
                    valid_overlap = min(prev_ref.shape[-1], curr_ref.shape[-1])
                    
                    if valid_overlap > 0:
                        shift = 0
                        if valid_overlap > search_range * 2:
                            shift = find_best_frame_shift(
                                torch.abs(prev_ref[..., :valid_overlap]), 
                                torch.abs(curr_ref[..., :valid_overlap]), 
                                search_range
                            )
                        start_off = max(0, min(-shift, search_range * 2))
                        aligned = complex_chunk[..., start_off:]
                        cross_len = min(valid_overlap, aligned.shape[-1])
                        
                        if cross_len > 0:
                            alpha = torch.linspace(0.0, 1.0, cross_len).to(device).view(1, 1, 1, cross_len)
                            merged = prev_ref[..., :cross_len] * (1 - alpha) + aligned[..., :cross_len] * alpha
                            full_complex_spec = torch.cat([
                                full_complex_spec[..., :-cross_len], merged, aligned[..., cross_len:]
                            ], dim=-1)
                        else:
                            full_complex_spec = torch.cat([full_complex_spec, aligned], dim=-1)
                    else:
                        full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
                
                next_expected_overlap = int(z_overlap_frames * ratio)
                if complex_chunk.shape[-1] >= next_expected_overlap:
                    prev_raw_tail = complex_chunk[..., -next_expected_overlap:]
                else:
                    prev_raw_tail = complex_chunk

            current_ph_idx += count
            current_z_frame = z_end_nominal
            if current_z_frame >= z.shape[2]: break

        if full_complex_spec is None: return np.array([]), 0.0
        return istft_finalize(net_g, full_complex_spec), (first_latency if first_latency else 0.0)


# ==========================================
# Cond 4: 全体一括 (Baseline)
# ==========================================
def synthesize_cond4_full(net_g, raw_text, sid, hps, phonemizer):
    """
    全体一括生成は「全生成が終わるまで何も出ない」ため、
    Latency = Total Processing Time となります。
    """
    device = next(net_g.parameters()).device
    sid = sid.to(device)
    
    # 簡易的に結合
    bunsetsu = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    full_text = " ".join(bunsetsu)
    stn_tst = get_text_from_phonemes(full_text, hps)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        process_start = time.time()
        
        audio = net_g.infer(
            x_tst, x_tst_lengths, sid=sid, 
            noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale
        )[0][0,0].data.cpu().float().numpy()
        
        if torch.cuda.is_available(): torch.cuda.synchronize()
        latency = time.time() - process_start
        
    return audio, latency

In [31]:
print("Starting measurement loop...")

# チャンクサイズ設定 (Cond 2, 3用)
mora_chunk = 1

for i, line in enumerate(lines):
    line = line.strip()
    if not line: continue
    parts = line.split("|")
    
    # 読み込み
    file_path = parts[0]
    spk_id_val = int(parts[1])
    raw_text_jp = parts[2]
    
    filename = os.path.basename(file_path).replace(".wav", "")
    sid = torch.LongTensor([spk_id_val]).to(device)
    
    print(f"\nProcessing: {filename} (SID: {spk_id_val})")
    print(f"Text: {raw_text_jp[:30]}...") # 長いので省略表示

    try:
        # Cond 1: 文節単位単純接続
        audio_c1 = measure_and_print(
            "Cond 1 (Naive Bunsetsu Concat)", 
            synthesize_cond1_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)

        # Cond 2: スペクトログラム単純接続
        audio_c2 = measure_and_print(
            f"Cond 2 (Spec Naive Concat, chunk={mora_chunk})", 
            synthesize_cond2_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond2.wav"), hps.data.sampling_rate, audio_c2)

        # Cond 3: 相互相関OLA
        audio_c3 = measure_and_print(
            f"Cond 3 (Correlation OLA, chunk={mora_chunk})", 
            synthesize_cond3_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer, 
            5, 2  # overlap=5, search=2
        )
        write(os.path.join(output_dir, f"{filename}_cond3.wav"), hps.data.sampling_rate, audio_c3)

        # Cond 4: 全体一括 (Baseline)
        audio_c4 = measure_and_print(
            "Cond 4 (Full Context Baseline)", 
            synthesize_cond4_full, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond4.wav"), hps.data.sampling_rate, audio_c4)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        import traceback
        traceback.print_exc()

print("All tasks finished.")

Starting measurement loop...

Processing: FJK_C051_118 (SID: 368)
Text: でも[なんかこう]そうすると、またまたあったかくなっちゃって...
--- Cond 1 (Naive Bunsetsu Concat) ---
Audio duration: 0.00 seconds
Elapsed time: 0.6090 seconds
Real Time Factor (RTF): 4871.9501
------------------------------
Error processing FJK_C051_118: 'tuple' object has no attribute 'dtype'

Processing: FJK_C051_170 (SID: 368)
Text: やっぱ、やっぱディーが最後かな...


Traceback (most recent call last):
  File "/tmp/ipykernel_340830/1984995445.py", line 29, in <module>
    write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)
  File "/Users/naru/.conda/envs/ms-istft2/lib/python3.9/site-packages/scipy/io/wavfile.py", line 771, in write
    dkind = data.dtype.kind
AttributeError: 'tuple' object has no attribute 'dtype'


--- Cond 1 (Naive Bunsetsu Concat) ---
Audio duration: 0.00 seconds
Elapsed time: 0.2037 seconds
Real Time Factor (RTF): 1629.5795
------------------------------
Error processing FJK_C051_170: 'tuple' object has no attribute 'dtype'

Processing: FKC_C031_002 (SID: 369)
Text: [うんとね]多分あたし一番持ってる気がするって感じ...


Traceback (most recent call last):
  File "/tmp/ipykernel_340830/1984995445.py", line 29, in <module>
    write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)
  File "/Users/naru/.conda/envs/ms-istft2/lib/python3.9/site-packages/scipy/io/wavfile.py", line 771, in write
    dkind = data.dtype.kind
AttributeError: 'tuple' object has no attribute 'dtype'


--- Cond 1 (Naive Bunsetsu Concat) ---
Audio duration: 0.00 seconds
Elapsed time: 0.3554 seconds
Real Time Factor (RTF): 2843.5745
------------------------------
Error processing FKC_C031_002: 'tuple' object has no attribute 'dtype'

Processing: FMS_C051_072 (SID: 370)
Text: そうだよね、うんうん...
--- Cond 1 (Naive Bunsetsu Concat) ---
Audio duration: 0.00 seconds
Elapsed time: 0.1026 seconds
Real Time Factor (RTF): 820.8351
------------------------------
Error processing FMS_C051_072: 'tuple' object has no attribute 'dtype'

Processing: FMT_C041_134 (SID: 371)
Text: 何だそりゃ...
--- Cond 1 (Naive Bunsetsu Concat) ---
Audio duration: 0.00 seconds
Elapsed time: 0.0711 seconds
Real Time Factor (RTF): 568.9774
------------------------------
Error processing FMT_C041_134: 'tuple' object has no attribute 'dtype'

Processing: FMT_C041_259 (SID: 371)
Text: ディーはエーより後ってことだよね...


...

Traceback (most recent call last):
  File "/tmp/ipykernel_340830/1984995445.py", line 29, in <module>
    write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)
  File "/Users/naru/.conda/envs/ms-istft2/lib/python3.9/site-packages/scipy/io/wavfile.py", line 771, in write
    dkind = data.dtype.kind
AttributeError: 'tuple' object has no attribute 'dtype'
Traceback (most recent call last):
  File "/tmp/ipykernel_340830/1984995445.py", line 29, in <module>
    write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)
  File "/Users/naru/.conda/envs/ms-istft2/lib/python3.9/site-packages/scipy/io/wavfile.py", line 771, in write
    dkind = data.dtype.kind
AttributeError: 'tuple' object has no attribute 'dtype'


--- Cond 1 (Naive Bunsetsu Concat) ---
Audio duration: 0.00 seconds
Elapsed time: 0.0990 seconds
Real Time Factor (RTF): 791.9903
------------------------------
Error processing FTS_C007_137: 'tuple' object has no attribute 'dtype'

Processing: FUE_C033_134 (SID: 378)
Text: [え]、わかんなくなってきた...
--- Cond 1 (Naive Bunsetsu Concat) ---
Audio duration: 0.00 seconds
Elapsed time: 0.1748 seconds
Real Time Factor (RTF): 1398.3002
------------------------------
Error processing FUE_C033_134: 'tuple' object has no attribute 'dtype'

Processing: FYH_C042_090 (SID: 379)
Text: この、お母さんがすごい、あんまり上品でない食べ方をしてるのね...


Traceback (most recent call last):
  File "/tmp/ipykernel_340830/1984995445.py", line 29, in <module>
    write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)
  File "/Users/naru/.conda/envs/ms-istft2/lib/python3.9/site-packages/scipy/io/wavfile.py", line 771, in write
    dkind = data.dtype.kind
AttributeError: 'tuple' object has no attribute 'dtype'
Traceback (most recent call last):
  File "/tmp/ipykernel_340830/1984995445.py", line 29, in <module>
    write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)
  File "/Users/naru/.conda/envs/ms-istft2/lib/python3.9/site-packages/scipy/io/wavfile.py", line 771, in write
    dkind = data.dtype.kind
AttributeError: 'tuple' object has no attribute 'dtype'


--- Cond 1 (Naive Bunsetsu Concat) ---
Audio duration: 0.00 seconds
Elapsed time: 0.3468 seconds
Real Time Factor (RTF): 2774.2958
------------------------------
Error processing FYH_C042_090: 'tuple' object has no attribute 'dtype'

Processing: FYH_C043_064 (SID: 379)
Text: はい、いいですか...
--- Cond 1 (Naive Bunsetsu Concat) ---
Audio duration: 0.00 seconds
Elapsed time: 0.0738 seconds
Real Time Factor (RTF): 590.5037
------------------------------
Error processing FYH_C043_064: 'tuple' object has no attribute 'dtype'
All tasks finished.


Traceback (most recent call last):
  File "/tmp/ipykernel_340830/1984995445.py", line 29, in <module>
    write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)
  File "/Users/naru/.conda/envs/ms-istft2/lib/python3.9/site-packages/scipy/io/wavfile.py", line 771, in write
    dkind = data.dtype.kind
AttributeError: 'tuple' object has no attribute 'dtype'
Traceback (most recent call last):
  File "/tmp/ipykernel_340830/1984995445.py", line 29, in <module>
    write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)
  File "/Users/naru/.conda/envs/ms-istft2/lib/python3.9/site-packages/scipy/io/wavfile.py", line 771, in write
    dkind = data.dtype.kind
AttributeError: 'tuple' object has no attribute 'dtype'


In [33]:
import time
import torch
import numpy as np
import os
from scipy.io.wavfile import write

# ==========================================
# 計測用ヘルパー関数の更新 (ここが重要)
# ==========================================
def measure_and_print(label, func, *args):
    """
    関数を実行し、レイテンシ等を計測して表示する。
    戻り値: 音声データ (numpy array) のみ
    """
    # GPU同期 (計測開始前)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start_time = time.time()
    
    # 関数の実行
    # synthesize関数は (audio, latency) を返すと想定
    result = func(*args)
    
    # 結果のアンパック (タプルかどうかの安全策を追加)
    if isinstance(result, tuple) and len(result) == 2:
        audio, first_latency = result
    else:
        # 万が一タプルでない場合 (古い関数など)
        audio = result
        first_latency = 0.0

    # GPU同期 (計測終了後)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    total_elapsed = time.time() - start_time
    
    # 音声の長さを計算
    if isinstance(audio, np.ndarray) and len(audio) > 0:
        duration = len(audio) / hps.data.sampling_rate
        rtf = total_elapsed / duration if duration > 0 else 0
    else:
        duration = 0
        rtf = 0

    print(f"--- {label} ---")
    print(f"Audio duration : {duration:.2f} sec")
    print(f"Total time     : {total_elapsed:.4f} sec")
    # レイテンシ表示 (ミリ秒)
    print(f"Latency (First): {first_latency * 1000:.2f} ms")
    print(f"RTF            : {rtf:.4f}")
    print("-" * 30)
    
    # ★重要: write関数でエラーにならないよう、音声データのみを返す
    return audio

# ==========================================
# メイン実行ループ
# ==========================================
print("Starting measurement loop (Latency fixed)...")

mora_chunk = 1

# エラーが出た行以降も処理できるよう try-except は維持
for i, line in enumerate(lines):
    line = line.strip()
    if not line: continue
    parts = line.split("|")
    
    file_path = parts[0]
    spk_id_val = int(parts[1])
    raw_text_jp = parts[2]
    
    filename = os.path.basename(file_path).replace(".wav", "")
    sid = torch.LongTensor([spk_id_val]).to(device)
    
    print(f"\nProcessing: {filename} (SID: {spk_id_val})")
    print(f"Text: {raw_text_jp[:30]}...")

    try:
        # Cond 1: 文節単位単純接続
        audio_c1 = measure_and_print(
            "Cond 1 (Naive Bunsetsu Concat)", 
            synthesize_cond1_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio_c1)

        # Cond 2: スペクトログラム単純接続
        audio_c2 = measure_and_print(
            f"Cond 2 (Spec Naive Concat, chunk={mora_chunk})", 
            synthesize_cond2_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond2.wav"), hps.data.sampling_rate, audio_c2)

        # Cond 3: 相互相関OLA
        audio_c3 = measure_and_print(
            f"Cond 3 (Correlation OLA, chunk={mora_chunk})", 
            synthesize_cond3_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer, 
            1, 2
        )
        write(os.path.join(output_dir, f"{filename}_cond3.wav"), hps.data.sampling_rate, audio_c3)

        # Cond 4: 全体一括 (Baseline)
        audio_c4 = measure_and_print(
            "Cond 4 (Full Context Baseline)", 
            synthesize_cond4_full, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond4.wav"), hps.data.sampling_rate, audio_c4)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        import traceback
        traceback.print_exc()

print("All tasks finished.")

Starting measurement loop (Latency fixed)...

Processing: FJK_C051_118 (SID: 368)
Text: でも[なんかこう]そうすると、またまたあったかくなっちゃって...
--- Cond 1 (Naive Bunsetsu Concat) ---
Audio duration : 7.58 sec
Total time     : 0.5712 sec
Latency (First): 66.68 ms
RTF            : 0.0753
------------------------------
--- Cond 2 (Spec Naive Concat, chunk=1) ---
Audio duration : 6.27 sec
Total time     : 0.4399 sec
Latency (First): 109.58 ms
RTF            : 0.0702
------------------------------
--- Cond 3 (Correlation OLA, chunk=1) ---
Audio duration : 6.28 sec
Total time     : 0.3562 sec
Latency (First): 60.00 ms
RTF            : 0.0567
------------------------------
--- Cond 4 (Full Context Baseline) ---
Audio duration : 5.87 sec
Total time     : 0.1864 sec
Latency (First): 183.92 ms
RTF            : 0.0317
------------------------------

Processing: FJK_C051_170 (SID: 368)
Text: やっぱ、やっぱディーが最後かな...
--- Cond 1 (Naive Bunsetsu Concat) ---
Audio duration : 2.32 sec
Total time     : 0.1684 sec
Latency (First): 

In [35]:
import time
import torch
import numpy as np
import math
import os
from scipy.io.wavfile import write

# ==========================================
# 1. データ蓄積用の辞書を初期化
# ==========================================
benchmark_data = {
    "Cond1": [],
    "Cond2": [],
    "Cond3": [],
    "Cond4": []
}

# ==========================================
# 2. 計測・集計用関数の定義
# ==========================================
def measure_and_collect(label, func, *args):
    """
    実行時間を計測し、音声データと統計情報を返す
    """
    # GPU同期 (計測開始前)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start_time = time.time()
    
    # 関数の実行: (audio, first_latency) を受け取る
    # 関数が古い形式(audioのみ)を返す場合に備えてチェック
    result = func(*args)
    
    if isinstance(result, tuple) and len(result) == 2:
        audio, first_latency = result
    else:
        audio = result
        first_latency = 0.0

    # GPU同期 (計測終了後)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    total_elapsed = time.time() - start_time
    
    # 指標計算
    if isinstance(audio, np.ndarray) and len(audio) > 0:
        duration = len(audio) / hps.data.sampling_rate
        rtf = total_elapsed / duration if duration > 0 else 0
    else:
        duration = 0
        rtf = 0
    
    # 統計データ辞書を作成
    stats = {
        "label": label,
        "duration": duration,
        "total_time": total_elapsed,
        "latency_ms": first_latency * 1000, # ms単位
        "rtf": rtf
    }

    # ログ表示
    print(f"--- {label} ---")
    print(f"Latency: {stats['latency_ms']:.2f} ms | RTF: {rtf:.4f}")
    
    return audio, stats

# ==========================================
# 3. 計測実行ループ
# ==========================================
print("Starting benchmark loop...")

mora_chunk = 1 # チャンクサイズ設定

# lines 変数は前段のコードで生成されたものを使用します
for i, line in enumerate(lines):
    line = line.strip()
    if not line: continue
    parts = line.split("|")
    
    file_path = parts[0]
    spk_id_val = int(parts[1])
    raw_text_jp = parts[2]
    
    filename = os.path.basename(file_path).replace(".wav", "")
    sid = torch.LongTensor([spk_id_val]).to(device)
    
    print(f"\n[{i+1}/{len(lines)}] {filename}")

    try:
        # --- Cond 1 ---
        audio, stats = measure_and_collect(
            "Cond 1", synthesize_cond1_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio)
        benchmark_data["Cond1"].append(stats)

        # --- Cond 2 ---
        audio, stats = measure_and_collect(
            "Cond 2", synthesize_cond2_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond2.wav"), hps.data.sampling_rate, audio)
        benchmark_data["Cond2"].append(stats)

        # --- Cond 3 ---
        audio, stats = measure_and_collect(
            "Cond 3", synthesize_cond3_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer, 5, 2
        )
        write(os.path.join(output_dir, f"{filename}_cond3.wav"), hps.data.sampling_rate, audio)
        benchmark_data["Cond3"].append(stats)

        # --- Cond 4 ---
        audio, stats = measure_and_collect(
            "Cond 4", synthesize_cond4_full, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond4.wav"), hps.data.sampling_rate, audio)
        benchmark_data["Cond4"].append(stats)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        import traceback
        traceback.print_exc()

# ==========================================
# 4. 集計結果の表示 (pandasなし版)
# ==========================================
print("\n" + "="*85)
print(f"{'BENCHMARK RESULTS (Mean ± Std)':^85}")
print("="*85)

# ヘッダー出力
# 幅指定: Cond:10, Latency:22, RTF:22, Time:15, Samples:8
header = f"{'Condition':<10} | {'Latency (ms)':<22} | {'RTF':<22} | {'Avg Time (s)':<15} | {'Samples':<8}"
print(header)
print("-" * 85)

def calculate_mean_std(values):
    """リストから平均と標準偏差(不偏分散)を計算するヘルパー関数"""
    n = len(values)
    if n == 0:
        return 0.0, 0.0
    
    mean = sum(values) / n
    
    if n == 1:
        return mean, 0.0
        
    variance = sum((x - mean) ** 2 for x in values) / (n - 1)
    std = math.sqrt(variance)
    return mean, std

for cond_name, stats_list in benchmark_data.items():
    if not stats_list:
        continue
    
    # データの抽出
    latencies = [s["latency_ms"] for s in stats_list]
    rtfs = [s["rtf"] for s in stats_list]
    times = [s["total_time"] for s in stats_list]
    
    # 計算
    lat_mean, lat_std = calculate_mean_std(latencies)
    rtf_mean, rtf_std = calculate_mean_std(rtfs)
    time_mean, _ = calculate_mean_std(times)
    
    n_samples = len(stats_list)
    
    # 文字列整形
    lat_str = f"{lat_mean:.2f} ± {lat_std:.2f}"
    rtf_str = f"{rtf_mean:.4f} ± {rtf_std:.4f}"
    time_str = f"{time_mean:.4f}"
    
    # 行出力
    row = f"{cond_name:<10} | {lat_str:<22} | {rtf_str:<22} | {time_str:<15} | {n_samples:<8}"
    print(row)

print("="*85)

Starting benchmark loop...

[1/16] FJK_C051_118
--- Cond 1 ---
Latency: 29.87 ms | RTF: 0.0729
--- Cond 2 ---
Latency: 87.57 ms | RTF: 0.0627
--- Cond 3 ---
Latency: 52.31 ms | RTF: 0.0571
--- Cond 4 ---
Latency: 186.41 ms | RTF: 0.0321

[2/16] FJK_C051_170
--- Cond 1 ---
Latency: 39.56 ms | RTF: 0.0691
--- Cond 2 ---
Latency: 90.95 ms | RTF: 0.0846
--- Cond 3 ---
Latency: 44.88 ms | RTF: 0.0799
--- Cond 4 ---
Latency: 82.01 ms | RTF: 0.0453

[3/16] FKC_C031_002
--- Cond 1 ---
Latency: 31.26 ms | RTF: 0.0755
--- Cond 2 ---
Latency: 39.68 ms | RTF: 0.0490
--- Cond 3 ---
Latency: 71.10 ms | RTF: 0.0736
--- Cond 4 ---
Latency: 144.51 ms | RTF: 0.0476

[4/16] FMS_C051_072
--- Cond 1 ---
Latency: 43.21 ms | RTF: 0.0863
--- Cond 2 ---
Latency: 46.97 ms | RTF: 0.0750
--- Cond 3 ---
Latency: 73.29 ms | RTF: 0.0941
--- Cond 4 ---
Latency: 51.66 ms | RTF: 0.0560

[5/16] FMT_C041_134
--- Cond 1 ---
Latency: 22.93 ms | RTF: 0.0616
--- Cond 2 ---
Latency: 26.49 ms | RTF: 0.0633
--- Cond 3 ---
Laten

In [36]:
import time
import torch
import numpy as np
import math
import os
from scipy.io.wavfile import write

# ==========================================
# 1. データ蓄積用の辞書を初期化
# ==========================================
benchmark_data = {
    "Cond1": [],
    "Cond2": [],
    "Cond3": [],
    "Cond4": []
}

# ==========================================
# 2. 計測・集計用関数の定義
# ==========================================
def measure_and_collect(label, func, *args):
    """
    実行時間を計測し、音声データと統計情報を返す
    """
    # GPU同期 (計測開始前)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start_time = time.time()
    
    # 関数の実行: (audio, first_latency) を受け取る
    result = func(*args)
    
    # アンパック
    if isinstance(result, tuple) and len(result) == 2:
        audio, first_latency = result
    else:
        audio = result
        first_latency = 0.0

    # GPU同期 (計測終了後)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    total_elapsed = time.time() - start_time
    
    # 指標計算
    if isinstance(audio, np.ndarray) and len(audio) > 0:
        duration = len(audio) / hps.data.sampling_rate
        rtf = total_elapsed / duration if duration > 0 else 0
    else:
        duration = 0
        rtf = 0
    
    # 統計データ
    latency_ms = first_latency * 1000
    
    stats = {
        "label": label,
        "duration": duration,
        "total_time": total_elapsed,
        "latency_ms": latency_ms,
        "rtf": rtf
    }

    # ログ表示 (Latency (First) を明示)
    print(f"--- {label} ---")
    print(f"Latency (First): {latency_ms:.2f} ms | RTF: {rtf:.4f}")
    
    return audio, stats

# ==========================================
# 3. 計測実行ループ
# ==========================================
print("Starting benchmark loop...")

mora_chunk = 1 

for i, line in enumerate(lines):
    line = line.strip()
    if not line: continue
    parts = line.split("|")
    
    file_path = parts[0]
    spk_id_val = int(parts[1])
    raw_text_jp = parts[2]
    
    filename = os.path.basename(file_path).replace(".wav", "")
    sid = torch.LongTensor([spk_id_val]).to(device)
    
    print(f"\n[{i+1}/{len(lines)}] {filename}")

    try:
        # Cond 1
        audio, stats = measure_and_collect(
            "Cond 1", synthesize_cond1_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio)
        benchmark_data["Cond1"].append(stats)

        # Cond 2
        audio, stats = measure_and_collect(
            "Cond 2", synthesize_cond2_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond2.wav"), hps.data.sampling_rate, audio)
        benchmark_data["Cond2"].append(stats)

        # Cond 3
        audio, stats = measure_and_collect(
            "Cond 3", synthesize_cond3_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer, 5, 2
        )
        write(os.path.join(output_dir, f"{filename}_cond3.wav"), hps.data.sampling_rate, audio)
        benchmark_data["Cond3"].append(stats)

        # Cond 4
        audio, stats = measure_and_collect(
            "Cond 4", synthesize_cond4_full, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond4.wav"), hps.data.sampling_rate, audio)
        benchmark_data["Cond4"].append(stats)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        import traceback
        traceback.print_exc()

# ==========================================
# 4. 集計結果の表示 (Mean & Variance)
# ==========================================
print("\n" + "="*95)
print(f"{'BENCHMARK RESULTS (Mean ± Std)':^95}")
print("="*95)

# ヘッダー
header = f"{'Condition':<10} | {'Latency (First) [ms]':<25} | {'RTF':<20} | {'Time [s]':<10} | {'N':<5}"
print(header)
print("-" * 95)

def calc_stats(values):
    """平均と標準偏差を計算"""
    n = len(values)
    if n == 0: return 0.0, 0.0
    mean = sum(values) / n
    if n == 1: return mean, 0.0
    var = sum((x - mean) ** 2 for x in values) / (n - 1)
    std = math.sqrt(var)
    return mean, std

for cond_name, stats_list in benchmark_data.items():
    if not stats_list: continue
    
    latencies = [s["latency_ms"] for s in stats_list]
    rtfs = [s["rtf"] for s in stats_list]
    times = [s["total_time"] for s in stats_list]
    
    l_mean, l_std = calc_stats(latencies)
    r_mean, r_std = calc_stats(rtfs)
    t_mean, t_std = calc_stats(times)
    
    # Latency (First) の平均±標準偏差
    lat_str = f"{l_mean:.2f} ± {l_std:.2f}"
    rtf_str = f"{r_mean:.4f} ± {r_std:.4f}"
    time_str = f"{t_mean:.3f}"
    
    print(f"{cond_name:<10} | {lat_str:<25} | {rtf_str:<20} | {time_str:<10} | {len(stats_list):<5}")

print("="*95)

Starting benchmark loop...

[1/16] FJK_C051_118
--- Cond 1 ---
Latency (First): 50.76 ms | RTF: 0.0652
--- Cond 2 ---
Latency (First): 62.55 ms | RTF: 0.0647
--- Cond 3 ---
Latency (First): 60.71 ms | RTF: 0.0609
--- Cond 4 ---
Latency (First): 221.54 ms | RTF: 0.0381

[2/16] FJK_C051_170
--- Cond 1 ---
Latency (First): 39.42 ms | RTF: 0.0689
--- Cond 2 ---
Latency (First): 70.97 ms | RTF: 0.0740
--- Cond 3 ---
Latency (First): 35.31 ms | RTF: 0.0564
--- Cond 4 ---
Latency (First): 89.39 ms | RTF: 0.0492

[3/16] FKC_C031_002
--- Cond 1 ---
Latency (First): 44.46 ms | RTF: 0.0747
--- Cond 2 ---
Latency (First): 80.89 ms | RTF: 0.0617
--- Cond 3 ---
Latency (First): 41.73 ms | RTF: 0.0587
--- Cond 4 ---
Latency (First): 136.65 ms | RTF: 0.0450

[4/16] FMS_C051_072
--- Cond 1 ---
Latency (First): 36.05 ms | RTF: 0.0729
--- Cond 2 ---
Latency (First): 42.04 ms | RTF: 0.0698
--- Cond 3 ---
Latency (First): 36.89 ms | RTF: 0.0592
--- Cond 4 ---
Latency (First): 37.79 ms | RTF: 0.0410

[5/16]

In [37]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def save_spectrogram_plot(audio, sampling_rate, output_path, title=None):
    """
    音声波形(numpy array)からスペクトログラムを計算し、画像として保存する
    """
    # 前処理: audioをtensor化
    if not isinstance(audio, torch.Tensor):
        audio_tensor = torch.FloatTensor(audio)
    else:
        audio_tensor = audio

    # STFT設定 (VITSの標準的な設定に合わせるか、一般的な設定を使用)
    n_fft = 1024
    hop_length = 256
    win_length = 1024
    window = torch.hann_window(win_length)
    
    # STFT計算
    # [1, T] -> [1, F, T]
    spec = torch.stft(
        audio_tensor.unsqueeze(0), 
        n_fft, 
        hop_length, 
        win_length, 
        window, 
        return_complex=True
    )
    spec = torch.abs(spec).squeeze(0) # [F, T]
    
    # デシベル変換 (20log10(|spec|))
    spec_db = 20 * torch.log10(torch.clamp(spec, min=1e-5))
    spec_db = spec_db.numpy()
    
    # プロット作成
    plt.figure(figsize=(12, 6))
    
    # スペクトログラム描画
    # origin='lower' で低周波を下にする
    plt.imshow(spec_db, aspect='auto', origin='lower', cmap='inferno',
               extent=[0, len(audio)/sampling_rate, 0, sampling_rate/2])
    
    plt.colorbar(format='%+2.0f dB')
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency (Hz)')
    
    if title:
        plt.title(title)
        
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close() # メモリ開放

In [38]:
# ... (前略: 計測関数の定義などはそのまま) ...

print("Starting benchmark loop with Spectrograms...")

mora_chunk = 1 

for i, line in enumerate(lines):
    line = line.strip()
    if not line: continue
    parts = line.split("|")
    
    file_path = parts[0]
    spk_id_val = int(parts[1])
    raw_text_jp = parts[2]
    
    filename = os.path.basename(file_path).replace(".wav", "")
    sid = torch.LongTensor([spk_id_val]).to(device)
    
    print(f"\n[{i+1}/{len(lines)}] {filename}")

    try:
        # --- Cond 1 ---
        audio, stats = measure_and_collect(
            "Cond 1", synthesize_cond1_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        # 音声保存
        write(os.path.join(output_dir, f"{filename}_cond1.wav"), hps.data.sampling_rate, audio)
        # ★スペクトログラム画像保存
        save_spectrogram_plot(
            audio, hps.data.sampling_rate, 
            os.path.join(output_dir, f"{filename}_cond1_spec.png"),
            title=f"Cond 1: Naive Bunsetsu (Latency: {stats['latency_ms']:.1f}ms)"
        )
        benchmark_data["Cond1"].append(stats)

        # --- Cond 2 ---
        audio, stats = measure_and_collect(
            "Cond 2", synthesize_cond2_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond2.wav"), hps.data.sampling_rate, audio)
        # ★スペクトログラム画像保存
        save_spectrogram_plot(
            audio, hps.data.sampling_rate, 
            os.path.join(output_dir, f"{filename}_cond2_spec.png"),
            title=f"Cond 2: Spec Naive (Latency: {stats['latency_ms']:.1f}ms)"
        )
        benchmark_data["Cond2"].append(stats)

        # --- Cond 3 ---
        audio, stats = measure_and_collect(
            "Cond 3", synthesize_cond3_bunsetsu_M, 
            net_g, raw_text_jp, sid, hps, phonemizer, 5, 2
        )
        write(os.path.join(output_dir, f"{filename}_cond3.wav"), hps.data.sampling_rate, audio)
        # ★スペクトログラム画像保存
        save_spectrogram_plot(
            audio, hps.data.sampling_rate, 
            os.path.join(output_dir, f"{filename}_cond3_spec.png"),
            title=f"Cond 3: Correlation OLA (Latency: {stats['latency_ms']:.1f}ms)"
        )
        benchmark_data["Cond3"].append(stats)

        # --- Cond 4 ---
        audio, stats = measure_and_collect(
            "Cond 4", synthesize_cond4_full, 
            net_g, raw_text_jp, sid, hps, phonemizer
        )
        write(os.path.join(output_dir, f"{filename}_cond4.wav"), hps.data.sampling_rate, audio)
        # ★スペクトログラム画像保存
        save_spectrogram_plot(
            audio, hps.data.sampling_rate, 
            os.path.join(output_dir, f"{filename}_cond4_spec.png"),
            title=f"Cond 4: Baseline (Latency: {stats['latency_ms']:.1f}ms)"
        )
        benchmark_data["Cond4"].append(stats)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        import traceback
        traceback.print_exc()

# ... (後略: 集計結果の表示コード) ...

Starting benchmark loop with Spectrograms...

[1/16] FJK_C051_118
--- Cond 1 ---
Latency (First): 56.06 ms | RTF: 0.0699
--- Cond 2 ---
Latency (First): 82.30 ms | RTF: 0.0569
--- Cond 3 ---
Latency (First): 52.55 ms | RTF: 0.0566
--- Cond 4 ---
Latency (First): 176.90 ms | RTF: 0.0305

[2/16] FJK_C051_170
--- Cond 1 ---
Latency (First): 31.18 ms | RTF: 0.0706
--- Cond 2 ---
Latency (First): 40.05 ms | RTF: 0.0532
--- Cond 3 ---
Latency (First): 36.96 ms | RTF: 0.0532
--- Cond 4 ---
Latency (First): 88.04 ms | RTF: 0.0484

[3/16] FKC_C031_002
--- Cond 1 ---
Latency (First): 78.87 ms | RTF: 0.0892
--- Cond 2 ---
Latency (First): 68.74 ms | RTF: 0.0681
--- Cond 3 ---
Latency (First): 48.80 ms | RTF: 0.0692
--- Cond 4 ---
Latency (First): 106.00 ms | RTF: 0.0350

[4/16] FMS_C051_072
--- Cond 1 ---
Latency (First): 43.66 ms | RTF: 0.0735
--- Cond 2 ---
Latency (First): 38.63 ms | RTF: 0.0943
--- Cond 3 ---
Latency (First): 44.29 ms | RTF: 0.0619
--- Cond 4 ---
Latency (First): 45.82 ms | R

In [39]:
# ==========================================
# Cond 1: 文節単位単純接続 (分割点返却版)
# ==========================================
def synthesize_cond1_bunsetsu_M_splits(net_g, raw_text, sid, hps, phonemizer):
    device = next(net_g.parameters()).device
    sid = sid.to(device)
    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    audio_segments = []
    first_latency = None
    process_start = time.time()
    
    for ph in bunsetsu_chunks:
        if not ph: continue
        stn_tst = get_text_from_phonemes(ph, hps)
        with torch.no_grad():
            x_tst = stn_tst.to(device).unsqueeze(0)
            x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
            audio = net_g.infer(
                x_tst, x_tst_lengths, sid=sid, 
                noise_scale=noise_scale, noise_scale_w=noise_scale_w, length_scale=length_scale
            )[0][0,0].data.cpu().float().numpy()
            
            if first_latency is None:
                if torch.cuda.is_available(): torch.cuda.synchronize()
                first_latency = time.time() - process_start

        audio_segments.append(audio)
    
    if not audio_segments: return np.array([]), 0.0, []
    
    # ★分割点の計算 (累積サンプル数)
    # [len1, len2, len3] -> [len1, len1+len2, len1+len2+len3] ... の最後の1つ手前まで
    lengths = [len(a) for a in audio_segments]
    split_points = np.cumsum(lengths)[:-1].tolist()
    
    return np.concatenate(audio_segments), (first_latency if first_latency else 0.0), split_points

# ==========================================
# Cond 2: スペクトログラム単純接続 (分割点返却版)
# ==========================================
def synthesize_cond2_bunsetsu_M_splits(net_g, raw_text, sid, hps, phonemizer):
    device = next(net_g.parameters()).device
    sid = sid.to(device)
    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    all_phoneme_ids = []
    chunk_counts = []
    for ph in bunsetsu_chunks:
        if not ph: continue
        ids = get_text_from_phonemes(ph, hps)
        chunk_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
        
    if not all_phoneme_ids: return np.array([]), 0.0, []

    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        
        process_start = time.time()
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, noise_scale, noise_scale_w, length_scale
        )
        w_ceil_flat = w_ceil.squeeze() 

        full_complex_spec = None
        split_points_frames = [] # フレーム単位の分割点
        
        current_ph_idx = 0
        current_z_frame = 0
        first_latency = None
        
        for i, count in enumerate(chunk_counts):
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            z_end_frame = current_z_frame + z_len
            if z_end_frame > z.shape[2]: z_end_frame = z.shape[2]
            z_chunk = z[:, :, current_z_frame : z_end_frame]
            
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                if first_latency is None:
                    if torch.cuda.is_available(): torch.cuda.synchronize()
                    first_latency = time.time() - process_start
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    # ★結合前のフレーム位置を記録
                    split_points_frames.append(full_complex_spec.shape[-1])
                    full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
            
            current_ph_idx += count
            current_z_frame = z_end_frame
            if current_z_frame >= z.shape[2]: break

        if full_complex_spec is None: return np.array([]), 0.0, []
        
        audio = istft_finalize(net_g, full_complex_spec)
        
        # フレーム -> サンプル数に変換
        hop_length = hps.data.hop_length
        split_points = [f * hop_length for f in split_points_frames]
        
        return audio, (first_latency if first_latency else 0.0), split_points

# ==========================================
# Cond 3: 相互相関OLA (分割点返却版)
# ==========================================
def synthesize_cond3_bunsetsu_M_splits(net_g, raw_text, sid, hps, phonemizer, z_overlap_frames=5, search_range=2):
    device = next(net_g.parameters()).device
    sid = sid.to(device)
    bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
    
    all_phoneme_ids = []
    chunk_counts = []
    for ph in bunsetsu_chunks:
        if not ph: continue
        ids = get_text_from_phonemes(ph, hps)
        chunk_counts.append(len(ids))
        all_phoneme_ids.extend(ids.tolist())
    
    if not all_phoneme_ids: return np.array([]), 0.0, []

    stn_tst = torch.LongTensor(all_phoneme_ids)
    
    with torch.no_grad():
        x_tst = stn_tst.to(device).unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).to(device)
        process_start = time.time()
        z, w_ceil, g = get_z_and_phoneme_durations(
            net_g, x_tst, x_tst_lengths, sid, noise_scale, noise_scale_w, length_scale
        )
        w_ceil_flat = w_ceil.squeeze()

        full_complex_spec = None
        split_points_frames = []
        
        prev_raw_tail = None
        current_ph_idx = 0
        current_z_frame = 0
        first_latency = None
        ratio = None 
        
        for count in chunk_counts:
            durations = w_ceil_flat[current_ph_idx : current_ph_idx + count]
            z_len = int(torch.sum(durations).item())
            z_end_nominal = current_z_frame + z_len
            z_end_decode = z_end_nominal + z_overlap_frames
            if z_end_decode > z.shape[2]: z_end_decode = z.shape[2]
            z_chunk = z[:, :, current_z_frame : z_end_decode]
            
            if z_chunk.shape[2] > 0:
                _, _, spec, phase = net_g.dec(z_chunk, g=g)
                complex_chunk = spec * torch.exp(1j * phase)
                
                if first_latency is None:
                    if torch.cuda.is_available(): torch.cuda.synchronize()
                    first_latency = time.time() - process_start
                if ratio is None:
                    ratio = complex_chunk.shape[-1] / z_chunk.shape[-1] if z_chunk.shape[-1] > 0 else 1.0
                
                # OLA処理
                actual_z_overlap = max(0, z_chunk.shape[-1] - z_len)
                spec_overlap_len = int(actual_z_overlap * ratio)
                
                if full_complex_spec is None:
                    full_complex_spec = complex_chunk
                else:
                    if prev_raw_tail is None: prev_ref = full_complex_spec[..., -spec_overlap_len:]
                    else: prev_ref = prev_raw_tail
                    curr_ref = complex_chunk[..., :spec_overlap_len]
                    valid_overlap = min(prev_ref.shape[-1], curr_ref.shape[-1])
                    
                    # ★結合処理
                    # OLAの場合はクロスフェードの中心付近をつなぎ目とみなしたいが、
                    # 簡易的に結合直前の末尾位置を記録する
                    current_end_pos = full_complex_spec.shape[-1]
                    # クロスフェードの長さ分だけ戻った位置が中心だが、
                    # ここでは「結合が起きた位置」として記録
                    
                    if valid_overlap > 0:
                        shift = 0
                        if valid_overlap > search_range * 2:
                            shift = find_best_frame_shift(torch.abs(prev_ref[..., :valid_overlap]), torch.abs(curr_ref[..., :valid_overlap]), search_range)
                        start_off = max(0, min(-shift, search_range * 2))
                        aligned = complex_chunk[..., start_off:]
                        cross_len = min(valid_overlap, aligned.shape[-1])
                        
                        if cross_len > 0:
                            alpha = torch.linspace(0.0, 1.0, cross_len).to(device).view(1, 1, 1, cross_len)
                            merged = prev_ref[..., :cross_len] * (1 - alpha) + aligned[..., :cross_len] * alpha
                            
                            # 結合位置の記録: 現在の長さ - クロスフェード長 + クロスフェード長/2 (中心)
                            split_points_frames.append(full_complex_spec.shape[-1] - cross_len + (cross_len // 2))
                            
                            full_complex_spec = torch.cat([full_complex_spec[..., :-cross_len], merged, aligned[..., cross_len:]], dim=-1)
                        else:
                            split_points_frames.append(full_complex_spec.shape[-1])
                            full_complex_spec = torch.cat([full_complex_spec, aligned], dim=-1)
                    else:
                        split_points_frames.append(full_complex_spec.shape[-1])
                        full_complex_spec = torch.cat([full_complex_spec, complex_chunk], dim=-1)
                
                next_expected_overlap = int(z_overlap_frames * ratio)
                if complex_chunk.shape[-1] >= next_expected_overlap: prev_raw_tail = complex_chunk[..., -next_expected_overlap:]
                else: prev_raw_tail = complex_chunk

            current_ph_idx += count
            current_z_frame = z_end_nominal
            if current_z_frame >= z.shape[2]: break

        if full_complex_spec is None: return np.array([]), 0.0, []
        audio = istft_finalize(net_g, full_complex_spec)
        hop_length = hps.data.hop_length
        split_points = [f * hop_length for f in split_points_frames]
        return audio, (first_latency if first_latency else 0.0), split_points

In [40]:
import numpy as np

def calculate_phoneme_stats(text_list, hps, phonemizer):
    """
    評価用テキストリストから、音素数の統計（平均・標準偏差）を算出する
    """
    
    # 統計用リスト
    chunk_lengths = []    # 文節ごとの音素数 (Cond 1, 2, 3用)
    sentence_lengths = [] # 文全体の音素数 (Cond 4用)
    
    print(f"Calculating stats for {len(text_list)} sentences...")
    
    for i, raw_text in enumerate(text_list):
        # 1. 文節分割して音素数をカウント
        try:
            # 既存の関数を使用 (文脈に合わせて関数名は調整してください)
            bunsetsu_chunks = get_bunsetsu_chunks_mecab(raw_text, phonemizer)
            
            # 文全体の音素IDリスト
            current_sent_ids = []
            
            for ph in bunsetsu_chunks:
                if not ph: continue
                # 音素列をIDに変換して長さを取得
                ids = get_text_from_phonemes(ph, hps)
                length = len(ids)
                
                if length > 0:
                    chunk_lengths.append(length)
                    current_sent_ids.extend(ids.tolist())
            
            # 文全体の長さ
            if len(current_sent_ids) > 0:
                sentence_lengths.append(len(current_sent_ids))
                
        except Exception as e:
            print(f"Error at index {i}: {e}")

    # --- 統計量の算出 ---
    print("\n" + "="*30)
    print(" ■ 音素数の統計結果")
    print("="*30)
    
    # 1. 文節単位 (Chunks)
    if chunk_lengths:
        chunk_mean = np.mean(chunk_lengths)
        chunk_std = np.std(chunk_lengths)
        chunk_min = np.min(chunk_lengths)
        chunk_max = np.max(chunk_lengths)
        
        print(f"【文節単位 (Cond 1-3)】")
        print(f"  - 平均 (Mean): {chunk_mean:.2f} 音素")
        print(f"  - 標準偏差 (Std): {chunk_std:.2f}")
        print(f"  - 最小/最大: {chunk_min} / {chunk_max}")
        print(f"  - 総文節数: {len(chunk_lengths)}")
    else:
        print("No chunk data found.")

    print("-" * 30)

    # 2. 文単位 (Sentences)
    if sentence_lengths:
        sent_mean = np.mean(sentence_lengths)
        sent_std = np.std(sentence_lengths)
        sent_min = np.min(sentence_lengths)
        sent_max = np.max(sentence_lengths)
        
        print(f"【文単位 (Cond 4)】")
        print(f"  - 平均 (Mean): {sent_mean:.2f} 音素")
        print(f"  - 標準偏差 (Std): {sent_std:.2f}")
        print(f"  - 最小/最大: {sent_min} / {sent_max}")
        print(f"  - 総文数: {len(sentence_lengths)}")
    else:
        print("No sentence data found.")
    print("="*30)

# --- 実行方法の例 ---
# 評価に使ったテキストリスト (例: test_text_list) を渡してください
# calculate_phoneme_stats(test_text_list, hps, phonemizer)

In [42]:
import re

# 1. 生データの定義
raw_input_text = """
uudb/tts1/data/test/wav/FJK_C051_118.wav
  Original: でも[なんかこう]そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた

uudb/tts1/data/test/wav/FJK_C051_170.wav
  Original: やっぱ、やっぱディーが最後かな

uudb/tts1/data/test/wav/FKC_C031_002.wav
  Original: [うんとね]多分あたし一番持ってる気がするって感じ

uudb/tts1/data/test/wav/FMS_C051_072.wav
  Original: そうだよね、うんうん

uudb/tts1/data/test/wav/FMT_C041_134.wav
  Original: 何だそりゃ

uudb/tts1/data/test/wav/FMT_C041_259.wav
  Original: ディーはエーより後ってことだよね

uudb/tts1/data/test/wav/FTH_C004_044.wav
  Original: [えっとね]じゃあシー、行くね

uudb/tts1/data/test/wav/FTH_C005_152.wav
  Original: [あ]じゃあ、最初は、エーの[その]、交番の外にサラリーマンがいて

uudb/tts1/data/test/wav/FTS_C002_107.wav
  Original: [あ]ちゃんと入ってないんだ、[あー]

uudb/tts1/data/test/wav/FTS_C002_175.wav
  Original: [あー]、オッケーオッケー

uudb/tts1/data/test/wav/FTS_C004_126.wav
  Original: そうね、えっみたいな感じだよね

uudb/tts1/data/test/wav/FTS_C006_050.wav
  Original: なるほど、わかりました

uudb/tts1/data/test/wav/FTS_C007_137.wav
  Original: [あー]、見てるんだ

uudb/tts1/data/test/wav/FUE_C033_134.wav
  Original: [え]、わかんなくなってきた

uudb/tts1/data/test/wav/FYH_C042_090.wav
  Original: この、お母さんがすごい、あんまり上品でない食べ方をしてるのね

uudb/tts1/data/test/wav/FYH_C043_064.wav
  Original: はい、いいですか
"""

# 2. テキスト部分の抽出とクリーニング
target_text_list = []
lines = raw_input_text.strip().split('\n')

for line in lines:
    if "Original:" in line:
        # "Original:" 以降の文字を取得
        text = line.split("Original:")[1].strip()
        
        # 必要であればここで [ ] などを削除しますが、
        # フィラー（あー、うんとね）も発話に含まれるため、そのまま音素化器に通します
        # text = re.sub(r'\[.*?\]', '', text) # もし[]の中身を消したい場合はコメントアウトを外す
        
        target_text_list.append(text)

print(f"抽出されたテキスト数: {len(target_text_list)}")
print("サンプル:", target_text_list[:3])

# 3. 統計量の算出 (calculate_phoneme_stats 関数を使用)
# hps, phonemizer は既存の環境変数を使用してください
calculate_phoneme_stats(target_text_list, hps, phonemizer)

抽出されたテキスト数: 16
サンプル: ['でも[なんかこう]そうすると、またまたあったかくなっちゃってこの長嶋監督が言ってるんだよねまた', 'やっぱ、やっぱディーが最後かな', '[うんとね]多分あたし一番持ってる気がするって感じ']
Calculating stats for 16 sentences...

 ■ 音素数の統計結果
【文節単位 (Cond 1-3)】
  - 平均 (Mean): 12.42 音素
  - 標準偏差 (Std): 5.95
  - 最小/最大: 3 / 27
  - 総文節数: 76
------------------------------
【文単位 (Cond 4)】
  - 平均 (Mean): 59.00 音素
  - 標準偏差 (Std): 41.72
  - 最小/最大: 20 / 176
  - 総文数: 16


In [43]:
print(device)

cpu
